In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score

import random
import os



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:


import pandas as pd

eeg_path = "/content/drive/MyDrive/data_n_back_test/eeg/eeg.parquet"

In [ ]:
df = pd.read_parquet(eeg_path)

In [ ]:
df = df[df["phase"] == 2].copy()

In [ ]:
eeg_cols = [
    'EEG.AF3','EEG.F7','EEG.F3','EEG.FC5',
    'EEG.T7','EEG.P7','EEG.O1','EEG.O2',
    'EEG.P8','EEG.T8','EEG.FC6','EEG.F4',
    'EEG.F8','EEG.AF4'
]

In [ ]:
WIN = 512
STRIDE = 512   # you can switch to 512 if you want no overlap

X_windows = []
y_windows = []
subjects = []

for (subj, test), g in df.groupby(["subject", "test"], sort=False):

    if "timestamp" in df.columns and "counter" in df.columns:
        g = g.sort_values(["timestamp", "counter"], kind="mergesort")
    elif "timestamp" in df.columns:
        g = g.sort_values("timestamp", kind="mergesort")
    else:
        g = g.sort_index()

    X = g[eeg_cols].to_numpy(dtype=np.float32)
    y = g["test"].to_numpy()

    if len(X) < WIN:
        continue

    for start in range(0, len(X) - WIN + 1, STRIDE):
        end = start + WIN
        X_windows.append(X[start:end].T)
        y_windows.append(y[start])
        subjects.append(subj)

X_windows = np.array(X_windows, dtype=np.float32)
y_windows = np.array(y_windows)
subjects  = np.array(subjects)

print("X_windows:", X_windows.shape)
print("y_windows:", y_windows.shape)
print("subjects :", subjects.shape)

X_windows: (15007, 14, 512)
y_windows: (15007,)
subjects : (15007,)


In [ ]:
import numpy as np

print("Starting bandpower computation")
print("X_windows dtype:", X_windows.dtype)
print("X_windows shape:", X_windows.shape)

FS = 128
bands = [
    ("theta", 4, 8),
    ("alpha", 8, 12),
    ("beta", 12, 30),
]

N, C, T = X_windows.shape
print("N:", N, "C:", C, "T:", T)

# ---- FFT frequencies ----
freqs = np.fft.rfftfreq(T, d=1.0/FS)
print("freqs shape:", freqs.shape)
print("First 10 freqs:", freqs[:10])
print("Last 10 freqs:", freqs[-10:])

# ---- FFT ----
X_fft = np.fft.rfft(X_windows, axis=-1)
print("X_fft shape:", X_fft.shape)
print("X_fft dtype:", X_fft.dtype)

# ---- PSD ----
PSD = (np.abs(X_fft) ** 2) / T
print("PSD shape:", PSD.shape)
print("PSD min/max:", PSD.min(), PSD.max())

# ---- Band extraction ----
bp_list = []

for name, f_lo, f_hi in bands:
    mask = (freqs >= f_lo) & (freqs < f_hi)

    print("\nBand:", name)
    print("Freq range:", f_lo, "-", f_hi)
    print("Number of bins in mask:", mask.sum())

    bp = PSD[..., mask]
    print("PSD slice shape:", bp.shape)

    bp_mean = bp.mean(axis=-1)
    print("After mean over freq axis:", bp_mean.shape)

    bp_list.append(bp_mean)

# ---- Stack bands ----
X_bp = np.stack(bp_list, axis=-1)
print("\nFinal stacked bandpower shape:", X_bp.shape)

# ---- Flatten ----
X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)
print("Flattened shape:", X_bp_flat.shape)

print("Done bandpower.")


Starting bandpower computation
X_windows dtype: float32
X_windows shape: (15007, 14, 512)
N: 15007 C: 14 T: 512
freqs shape: (257,)
First 10 freqs: [0.   0.25 0.5  0.75 1.   1.25 1.5  1.75 2.   2.25]
Last 10 freqs: [61.75 62.   62.25 62.5  62.75 63.   63.25 63.5  63.75 64.  ]
X_fft shape: (15007, 14, 257)
X_fft dtype: complex64
PSD shape: (15007, 14, 257)
PSD min/max: 0.0 13989107000.0

Band: theta
Freq range: 4 - 8
Number of bins in mask: 16
PSD slice shape: (15007, 14, 16)
After mean over freq axis: (15007, 14)

Band: alpha
Freq range: 8 - 12
Number of bins in mask: 16
PSD slice shape: (15007, 14, 16)
After mean over freq axis: (15007, 14)

Band: beta
Freq range: 12 - 30
Number of bins in mask: 72
PSD slice shape: (15007, 14, 72)
After mean over freq axis: (15007, 14)

Final stacked bandpower shape: (15007, 14, 3)
Flattened shape: (15007, 42)
Done bandpower.


In [ ]:
# X_bp currently shape: (15007, 14, 3)

print("Before relative normalization:")
print("Example band sums (first window, first channel):",
      X_bp[0, 0].sum())

# Relative within each channel
X_bp = X_bp / (X_bp.sum(axis=-1, keepdims=True) + 1e-8)

print("After relative normalization:")
print("Example band sums (first window, first channel):",
      X_bp[0, 0].sum())

# Flatten
N = X_bp.shape[0]
X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)

print("X_bp_flat shape:", X_bp_flat.shape)

Before relative normalization:
Example band sums (first window, first channel): 7497.5425
After relative normalization:
Example band sums (first window, first channel): 1.0
X_bp_flat shape: (15007, 42)


In [ ]:
import torch

X_time_t = torch.tensor(X_windows, dtype=torch.float32)
X_bp_t   = torch.tensor(X_bp_flat, dtype=torch.float32)

# Encode labels 0..2
unique_labels = np.unique(y_windows)
label_map = {v:i for i,v in enumerate(unique_labels)}
y_encoded = np.vectorize(label_map.get)(y_windows).astype(np.int64)

y_t = torch.tensor(y_encoded, dtype=torch.long)

print("X_time_t:", X_time_t.shape)
print("X_bp_t:", X_bp_t.shape)
print("y_t:", y_t.shape)
print("Label mapping:", label_map)

X_time_t: torch.Size([15007, 14, 512])
X_bp_t: torch.Size([15007, 42])
y_t: torch.Size([15007])
Label mapping: {np.int64(1): 0, np.int64(2): 1, np.int64(3): 2}


In [ ]:
# Subjects to remove
bad_subjects = ["subject_05", "subject_11", "subject_16"]

print("Original total windows:", len(subjects))

mask = ~np.isin(subjects, bad_subjects)

X_time_t = X_time_t[mask]
X_bp_t   = X_bp_t[mask]
y_t      = y_t[mask]
subjects = subjects[mask]

print("After removal total windows:", len(subjects))
print("Remaining unique subjects:", np.unique(subjects))
print("Remaining count:", len(np.unique(subjects)))

Original total windows: 15007
After removal total windows: 12159
Remaining unique subjects: ['subject_01' 'subject_02' 'subject_03' 'subject_04' 'subject_06'
 'subject_07' 'subject_08' 'subject_09' 'subject_10' 'subject_12'
 'subject_13' 'subject_14' 'subject_15']
Remaining count: 13


In [ ]:
def supcon_diff_subject_loss(proj, y, subj, temperature=0.1):
    """
    proj: (B, D) projection vectors
    y:    (B,) labels
    subj: (B,) subject ids (can be ints)
    positives: same y AND different subj
    """
    proj = F.normalize(proj, dim=1)
    B = proj.size(0)

    sim = (proj @ proj.t()) / temperature            # (B,B)
    sim = sim - sim.max(dim=1, keepdim=True).values  # stabilize

    y = y.view(-1, 1)
    subj = subj.view(-1, 1)

    same_y = (y == y.t())
    diff_s = (subj != subj.t())
    pos_mask = same_y & diff_s

    # exclude self
    eye = torch.eye(B, device=proj.device, dtype=torch.bool)
    pos_mask = pos_mask & (~eye)
    neg_mask = ~eye

    exp_sim = torch.exp(sim) * neg_mask
    denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

    # If an anchor has 0 positives in batch, we skip it (loss=0 for that anchor)
    pos_counts = pos_mask.sum(dim=1)  # (B,)
    log_prob = sim - torch.log(denom)

    # mean log-prob over positives
    loss_per_i = torch.zeros(B, device=proj.device)
    valid = pos_counts > 0
    loss_per_i[valid] = -(log_prob * pos_mask.float()).sum(dim=1)[valid] / pos_counts[valid].float()

    return loss_per_i[valid].mean() if valid.any() else torch.tensor(0.0, device=proj.device)


In [ ]:

N, C, T = X_windows.shape

cov_feats = []

for i in range(N):
    X = X_windows[i]                # (14, 512)
    X = X - X.mean(axis=1, keepdims=True)  # zero-mean per channel
    cov = (X @ X.T) / (T - 1)      # (14,14)

    # take upper triangle including diagonal
    iu = np.triu_indices(C)
    cov_vec = cov[iu]              # 105 features
    cov_feats.append(cov_vec)

X_cov = np.array(cov_feats, dtype=np.float32)

print("X_cov shape:", X_cov.shape)  # should be (N, 105)

X_cov shape: (15007, 105)


In [ ]:
X_cov_t = torch.tensor(X_cov, dtype=torch.float32)

In [ ]:
all_subjects = np.unique(subjects)
print("\nSubjects list:", all_subjects)


Subjects list: ['subject_01' 'subject_02' 'subject_03' 'subject_04' 'subject_06'
 'subject_07' 'subject_08' 'subject_09' 'subject_10' 'subject_12'
 'subject_13' 'subject_14' 'subject_15']


In [ ]:
# import numpy as np
# from torch.utils.data import Dataset

# class EEGDataset(Dataset):
#     def __init__(self, X_time, X_bp, X_cov, X_plv, X_roi, y, subj_strs):
#         self.X_time = X_time
#         self.X_bp   = X_bp
#         self.X_cov  = X_cov
#         self.X_plv  = X_plv
#         self.X_roi  = X_roi
#         self.y      = y

#         uniq = np.unique(subj_strs)
#         self.subj_map = {s: i for i, s in enumerate(uniq)}
#         self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

#     def __len__(self):
#         return len(self.y)

#     def __getitem__(self, idx):
#         return (
#             self.X_time[idx],
#             self.X_bp[idx],
#             self.X_cov[idx],
#             self.X_plv[idx],
#             self.X_roi[idx],
#             self.y[idx],
#             self.subj[idx]
#         )

In [ ]:
from torch.utils.data import Sampler
import random
from collections import defaultdict

class SubjectBalancedBatchSampler(Sampler):
    """
    Creates batches containing samples from multiple subjects.
    Works with your existing EEGDataset (which has .subj tensor).
    """

    def __init__(self, dataset, batch_size, subjects_per_batch=4):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_batch = subjects_per_batch

        # Build index list per subject
        self.subj_to_indices = defaultdict(list)
        for idx, s in enumerate(dataset.subj.tolist()):
            self.subj_to_indices[s].append(idx)

        self.subjects = list(self.subj_to_indices.keys())

        # samples per subject inside batch
        self.samples_per_subject = batch_size // subjects_per_batch

    def __iter__(self):
        # Shuffle indices per subject
        for s in self.subjects:
            random.shuffle(self.subj_to_indices[s])

        # Create subject iterators
        subj_iters = {s: iter(self.subj_to_indices[s]) for s in self.subjects}

        while True:
            # randomly choose subjects for this batch
            chosen_subjects = random.sample(self.subjects, self.subjects_per_batch)

            batch = []

            for s in chosen_subjects:
                for _ in range(self.samples_per_subject):
                    try:
                        batch.append(next(subj_iters[s]))
                    except StopIteration:
                        return  # stop when any subject runs out

            if len(batch) == self.batch_size:
                yield batch
            else:
                return

    def __len__(self):
        # approximate number of batches
        min_samples = min(len(v) for v in self.subj_to_indices.values())
        return (min_samples // self.samples_per_subject)

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt, hilbert

fs = 128
low, high = 4, 7

# Bandpass filter design
b, a = butter(4, [low/(fs/2), high/(fs/2)], btype='band')

N, C, T = X_windows.shape
plv_feats = []

for i in range(N):
    X = X_windows[i]  # (14, T)

    # Bandpass filter each channel
    X_filt = np.array([filtfilt(b, a, X[ch]) for ch in range(C)])

    # Hilbert transform → analytic signal
    analytic = hilbert(X_filt)
    phase = np.angle(analytic)  # (14, T)

    # Compute PLV matrix
    plv_mat = np.zeros((C, C))
    for ch1 in range(C):
        for ch2 in range(ch1, C):
            phase_diff = phase[ch1] - phase[ch2]
            plv = np.abs(np.mean(np.exp(1j * phase_diff)))
            plv_mat[ch1, ch2] = plv
            plv_mat[ch2, ch1] = plv

    # Upper triangle including diagonal
    iu = np.triu_indices(C)
    plv_vec = plv_mat[iu]  # 105 dims
    plv_feats.append(plv_vec)

X_plv = np.array(plv_feats, dtype=np.float32)
print("X_plv shape:", X_plv.shape)  # (N, 105)

X_plv_t = torch.tensor(X_plv, dtype=torch.float32)

X_plv shape: (15007, 105)


In [ ]:
# ----------------------------------------
# Alpha (8–12 Hz) PLV
# ----------------------------------------

low_a, high_a = 8, 12
b_a, a_a = butter(4, [low_a/(fs/2), high_a/(fs/2)], btype='band')

plv_alpha_feats = []

for i in range(N):
    X = X_windows[i]  # (14, T)

    # Bandpass filter
    X_filt = np.array([filtfilt(b_a, a_a, X[ch]) for ch in range(C)])

    analytic = hilbert(X_filt)
    phase = np.angle(analytic)

    plv_mat = np.zeros((C, C))
    for ch1 in range(C):
        for ch2 in range(ch1, C):
            phase_diff = phase[ch1] - phase[ch2]
            plv = np.abs(np.mean(np.exp(1j * phase_diff)))
            plv_mat[ch1, ch2] = plv
            plv_mat[ch2, ch1] = plv

    iu = np.triu_indices(C)
    plv_alpha_feats.append(plv_mat[iu])

X_plv_alpha = np.array(plv_alpha_feats, dtype=np.float32)
print("Alpha PLV shape:", X_plv_alpha.shape)

Alpha PLV shape: (15007, 105)


In [ ]:
# ----------------------------------------
# Combine Theta + Alpha PLV
# ----------------------------------------

X_plv_combined = np.concatenate([X_plv, X_plv_alpha], axis=1)
print("Combined PLV shape:", X_plv_combined.shape)  # (N, 210)

X_plv_t = torch.tensor(X_plv_combined, dtype=torch.float32)

Combined PLV shape: (15007, 210)


In [ ]:
def extract_embeddings(model, loader, device):

    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    E = torch.cat(E, dim=0)
    Y = torch.cat(Y, dim=0)

    return E, Y

In [ ]:
import torch.nn as nn

def set_partial_bn_adapt(model: nn.Module, allow=("time_branch",)):
    """
    Put BN layers in *specified submodules* into train mode (update running stats).
    Keep all other BN in eval mode.
    Always keep Dropout in eval mode.
    """
    # Default everything to eval
    model.eval()

    # Dropout always eval
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()

    # Enable BN train only in allowed named submodules
    for name, module in model.named_modules():
        if any(name.startswith(a) or (f".{a}." in name) or name.endswith(a) for a in allow):
            for m in module.modules():
                if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                    m.train()

In [ ]:
def prototype_loss(emb, y):
    """
    emb: (B, D) embedding vectors (from encoder)
    y:   (B,) class labels
    """

    B, D = emb.shape
    loss = 0.0
    count = 0

    classes = torch.unique(y)

    for c in classes:
        mask = (y == c)
        if mask.sum() < 2:
            continue

        class_emb = emb[mask]              # (Nc, D)
        proto = class_emb.mean(dim=0)      # (D,)

        loss += ((class_emb - proto)**2).sum(dim=1).mean()
        count += 1

    if count > 0:
        return loss / count
    else:
        return torch.tensor(0.0, device=emb.device)

In [ ]:
def supcon_diff_subject_loss(proj, y, subj, temperature=0.1):
    """
    proj: (B, D) projection vectors
    y:    (B,) labels
    subj: (B,) subject ids (can be ints)
    positives: same y AND different subj
    """
    proj = F.normalize(proj, dim=1)
    B = proj.size(0)

    sim = (proj @ proj.t()) / temperature            # (B,B)
    sim = sim - sim.max(dim=1, keepdim=True).values  # stabilize

    y = y.view(-1, 1)
    subj = subj.view(-1, 1)

    same_y = (y == y.t())
    diff_s = (subj != subj.t())
    pos_mask = same_y & diff_s

    # exclude self
    eye = torch.eye(B, device=proj.device, dtype=torch.bool)
    pos_mask = pos_mask & (~eye)
    neg_mask = ~eye

    exp_sim = torch.exp(sim) * neg_mask
    denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

    # If an anchor has 0 positives in batch, we skip it (loss=0 for that anchor)
    pos_counts = pos_mask.sum(dim=1)  # (B,)
    log_prob = sim - torch.log(denom)

    # mean log-prob over positives
    loss_per_i = torch.zeros(B, device=proj.device)
    valid = pos_counts > 0
    loss_per_i[valid] = -(log_prob * pos_mask.float()).sum(dim=1)[valid] / pos_counts[valid].float()

    return loss_per_i[valid].mean() if valid.any() else torch.tensor(0.0, device=proj.device)


In [ ]:
import torch

def build_roi_workload_features_from_bp(
    X_bp_flat: torch.Tensor,          # (N, 42)
    n_ch: int = 14,
    bands=("theta", "alpha", "beta"),  # must match how you built X_bp
    frontal_idx=(0, 1, 2, 3),          # CHANGE THIS to match your montage
    parietal_idx=(10, 11, 12, 13),     # CHANGE THIS to match your montage
    use_log: bool = True,
    use_relative: bool = False,        # relative bandpower within channel (theta/alpha/beta)
    eps: float = 1e-8
) -> torch.Tensor:
    """
    Returns ROI features (N, 3):
      [frontal_theta, parietal_alpha, theta_alpha_ratio]
    """
    assert X_bp_flat.ndim == 2, "X_bp_flat must be (N, 42)"
    n_bands = len(bands)
    assert X_bp_flat.shape[1] == n_ch * n_bands, "bp dim mismatch"

    # reshape -> (N, ch, band)
    X = X_bp_flat.view(-1, n_ch, n_bands)

    # optional relative power within channel
    if use_relative:
        denom = X.sum(dim=2, keepdim=True).clamp_min(eps)
        X = X / denom

    # optional log (common for EEG power)
    if use_log:
        X = torch.log(X.clamp_min(eps))

    # band indices
    band_to_i = {b: i for i, b in enumerate(bands)}
    theta_i = band_to_i["theta"]
    alpha_i = band_to_i["alpha"]

    theta_ch = X[:, :, theta_i]  # (N, ch)
    alpha_ch = X[:, :, alpha_i]  # (N, ch)

    frontal_theta = theta_ch[:, list(frontal_idx)].mean(dim=1)   # (N,)
    parietal_alpha = alpha_ch[:, list(parietal_idx)].mean(dim=1) # (N,)

    # ratio: theta_frontal / alpha_parietal
    # If using log-space, ratio becomes (log theta - log alpha) which is great + stable.
    if use_log:
        theta_alpha_ratio = frontal_theta - parietal_alpha
    else:
        theta_alpha_ratio = frontal_theta / parietal_alpha.clamp_min(eps)

    roi = torch.stack([frontal_theta, parietal_alpha, theta_alpha_ratio], dim=1)  # (N,3)
    return roi

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FusionEncoder(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=42, roi_dim=3):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim   # 42 + 3 = 45

        # ------------------
        # TIME BRANCH
        # ------------------
        self.time_branch = nn.Sequential(
            nn.Conv1d(14, 14, kernel_size=7, padding=3, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 14, kernel_size=15, padding=7, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        # ------------------
        # BANDPOWER + ROI BRANCH
        # ------------------
        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),   # was 42 → now 45
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        # ------------------
        # COVARIANCE BRANCH
        # ------------------
        self.cov_branch = nn.Sequential(
            nn.Linear(105, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32)
        )

        # ------------------
        # PLV BRANCH
        # ------------------
        self.plv_branch = nn.Sequential(
            nn.Linear(210, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32)
        )

        # ------------------
        # FUSION
        # ------------------
        self.embed = nn.Sequential(
            nn.Linear(32 + 32 + 32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi):

        # ------------------
        # Time branch
        # ------------------
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)  # (B,32)

        # ------------------
        # Bandpower + ROI
        # ------------------
        bp_in = torch.cat([x_bp, x_roi], dim=1)  # (B,45)
        z_bp = self.bp_branch(bp_in)

        # ------------------
        # Covariance
        # ------------------
        z_cov = self.cov_branch(x_cov)

        # ------------------
        # PLV
        # ------------------
        z_plv = self.plv_branch(x_plv)

        # ------------------
        # Fusion
        # ------------------
        z = torch.cat([z_time, z_bp, z_cov, z_plv], dim=1)
        e = self.embed(z)

        return e

In [ ]:
# import numpy as np
# from torch.utils.data import Dataset

# class EEGDataset(Dataset):
#     def __init__(self, X_time, X_bp, X_cov, X_plv, X_roi, y, subj_strs):
#         self.X_time = X_time
#         self.X_bp   = X_bp
#         self.X_cov  = X_cov
#         self.X_plv  = X_plv
#         self.X_roi  = X_roi
#         self.y      = y

#         uniq = np.unique(subj_strs)
#         self.subj_map = {s: i for i, s in enumerate(uniq)}
#         self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

#     def __len__(self):
#         return len(self.y)

#     def __getitem__(self, idx):
#         return (
#             self.X_time[idx],
#             self.X_bp[idx],
#             self.X_cov[idx],
#             self.X_plv[idx],
#             self.X_roi[idx],
#             self.y[idx],
#             self.subj[idx]
#         )

In [ ]:
import torch.nn as nn

class ContrastiveModel(nn.Module):
    def __init__(self,
                 emb_dim=64,
                 proj_dim=64,
                 bp_dim=42,
                 roi_dim=3):

        super().__init__()

        # Main encoder (learns embedding geometry)
        self.encoder = FusionEncoder(
            emb_dim=emb_dim,
            bp_dim=bp_dim,
            roi_dim=roi_dim
        )

        # Projection head for SupCon loss
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi):
        # Embedding
        e = self.encoder(x_time, x_bp, x_cov, x_plv, x_roi)

        # Projection (used only for contrastive loss)
        p = self.proj(e)

        return e, p

In [ ]:
import torch.nn.functional as F

def prototype_ce_loss(e, y, temperature=0.1):
    """
    e: (B, D) embeddings
    y: (B,) labels
    """

    classes = torch.unique(y)
    prototypes = []

    for c in classes:
        prototypes.append(e[y == c].mean(dim=0))

    prototypes = torch.stack(prototypes)  # (C, D)

    # normalize for cosine
    e_norm = F.normalize(e, dim=1)
    proto_norm = F.normalize(prototypes, dim=1)

    logits = e_norm @ proto_norm.T  # (B, C)
    logits = logits / temperature

    # map labels to [0, C-1]
    label_map = {c.item(): i for i, c in enumerate(classes)}
    y_mapped = torch.tensor([label_map[int(lbl.item())] for lbl in y],
                            device=y.device)

    loss = F.cross_entropy(logits, y_mapped)
    return loss

In [ ]:
'''few shot with 30 '''

'few shot with 30 '

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 33
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    # X_cov_train = X_cov_t[train_idx].clone()
    # X_cov_test  = X_cov_t[test_idx].clone()

    # cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    # cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    # X_cov_train = (X_cov_train - cov_mu) / cov_sd
    # X_cov_test  = (X_cov_test  - cov_mu) / cov_sd
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # # =========================
    # X_roi_train = build_roi_workload_features_from_bp(
    #     X_bp_train,
    #     # frontal_idx=(0,1,2,3),
    #     frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )
    # X_roi_test = build_roi_workload_features_from_bp(
    #     X_bp_test,
    #     frontal_idx=(0,1,2,3),
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )

    # roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    # roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    # X_roi_train = (X_roi_train - roi_mu) / roi_sd
    # X_roi_test  = (X_roi_test  - roi_mu) / roi_sd
    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # =====================================
    # FEATURE RELIABILITY WEIGHTING
    # =====================================
    # compute variance of each embedding dimension
    var = e_support.var(dim=0)
    # reliability = inverse variance
    weights = 1.0 / (var + 1e-6)
    # normalize weights
    weights = weights / weights.mean()
    # apply weighting
    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6120 | best 0.6120
Epoch 02 | zero-shot acc 0.6856 | best 0.6856
Epoch 03 | zero-shot acc 0.7077 | best 0.7077
Epoch 04 | zero-shot acc 0.6961 | best 0.7077
Epoch 05 | zero-shot acc 0.7150 | best 0.7150
Epoch 06 | zero-shot acc 0.7466 | best 0.7466
Epoch 07 | zero-shot acc 0.7487 | best 0.7487
Epoch 08 | zero-shot acc 0.7340 | best 0.7487
Epoch 09 | zero-shot acc 0.6961 | best 0.7487
Epoch 10 | zero-shot acc 0.7277 | best 0.7487
AdaBN + Mahalanobis few-shot acc: 0.8133803009986877

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4324 | best 0.4324
Epoch 02 | zero-shot acc 0.5144 | best 0.5144
Epoch 03 | zero-shot acc 0.5847 | best 0.5847
Epoch 04 | zero-shot acc 0.5836 | best 0.5847
Epoch 05 | zero-shot acc 0.5389 | best 0.5847
Epoch 06 | zero-shot acc 0.4920 | best 0.5847
Epoch 07 | zero-shot acc 0.5708 | best 0.5847
Epoch 08 | zero-shot acc 0.5751 | best 0.5847
Epoch 09 | zero-shot acc 0.6038 | best 0.

In [ ]:
'''few shot with 20 '''

'few shot with 20 '

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    # X_cov_train = X_cov_t[train_idx].clone()
    # X_cov_test  = X_cov_t[test_idx].clone()

    # cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    # cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    # X_cov_train = (X_cov_train - cov_mu) / cov_sd
    # X_cov_test  = (X_cov_test  - cov_mu) / cov_sd
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # # =========================
    # X_roi_train = build_roi_workload_features_from_bp(
    #     X_bp_train,
    #     # frontal_idx=(0,1,2,3),
    #     frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )
    # X_roi_test = build_roi_workload_features_from_bp(
    #     X_bp_test,
    #     frontal_idx=(0,1,2,3),
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )

    # roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    # roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    # X_roi_train = (X_roi_train - roi_mu) / roi_sd
    # X_roi_test  = (X_roi_test  - roi_mu) / roi_sd
    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # =====================================
    # FEATURE RELIABILITY WEIGHTING
    # =====================================
    # compute variance of each embedding dimension
    var = e_support.var(dim=0)
    # reliability = inverse variance
    weights = 1.0 / (var + 1e-6)
    # normalize weights
    weights = weights / weights.mean()
    # apply weighting
    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6172 | best 0.6172
Epoch 02 | zero-shot acc 0.6320 | best 0.6320
Epoch 03 | zero-shot acc 0.6709 | best 0.6709
Epoch 04 | zero-shot acc 0.6887 | best 0.6887
Epoch 05 | zero-shot acc 0.7382 | best 0.7382
Epoch 06 | zero-shot acc 0.7045 | best 0.7382
Epoch 07 | zero-shot acc 0.7150 | best 0.7382
Epoch 08 | zero-shot acc 0.7140 | best 0.7382
Epoch 09 | zero-shot acc 0.7066 | best 0.7382
Epoch 10 | zero-shot acc 0.7234 | best 0.7382
AdaBN + Mahalanobis few-shot acc: 0.842873215675354

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.5037 | best 0.5037
Epoch 02 | zero-shot acc 0.5133 | best 0.5133
Epoch 03 | zero-shot acc 0.4963 | best 0.5133
Epoch 04 | zero-shot acc 0.5091 | best 0.5133
Epoch 05 | zero-shot acc 0.4654 | best 0.5133
Epoch 06 | zero-shot acc 0.5442 | best 0.5442
Epoch 07 | zero-shot acc 0.5580 | best 0.5580
Epoch 08 | zero-shot acc 0.5847 | best 0.5847
Epoch 09 | zero-shot acc 0.6337 | best 0.6

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# ---------------------------
# LOAD
# ---------------------------
heat_eeg_path = "/content/drive/MyDrive/hc_eeg.parquet"
df_heat = pd.read_parquet(heat_eeg_path)


In [ ]:
# =========================
# EEG CHANNELS (LOCKED ORDER)
# =========================
eeg_cols_htc = [
    'EEG.AF3','EEG.F7','EEG.F3','EEG.FC5',
    'EEG.T7','EEG.P7','EEG.O1','EEG.O2',
    'EEG.P8','EEG.T8','EEG.FC6','EEG.F4',
    'EEG.F8','EEG.AF4'
]

# =========================
# FILTER TASK PHASE
# =========================
df_htc = df_heat[df_heat["phase"] == 2].copy()

# labels: 1→0, 2→1
df_htc["label"] = df_htc["test"] - 1

print("Filtered df_htc:", df_htc.shape)

Filtered df_htc: (2597172, 130)


In [ ]:
WIN_HTC = 512
STRIDE_HTC = 512

X_windows_htc = []
y_windows_htc = []
subjects_htc = []

for (subj, test), g in df_htc.groupby(["subject", "test"], sort=False):

    g = g.sort_values(["timestamp", "EEG.Counter"], kind="mergesort")

    X = g[eeg_cols_htc].to_numpy(dtype=np.float32)
    y = g["label"].to_numpy()

    if len(X) < WIN_HTC:
        continue

    for start in range(0, len(X) - WIN_HTC + 1, STRIDE_HTC):
        end = start + WIN_HTC

        X_windows_htc.append(X[start:end].T)  # (14, 512)
        y_windows_htc.append(y[start])
        subjects_htc.append(subj)

X_windows_htc = np.array(X_windows_htc, dtype=np.float32)
y_windows_htc = np.array(y_windows_htc)
subjects_htc  = np.array(subjects_htc)

print("X_windows_htc:", X_windows_htc.shape)
print("y_windows_htc:", y_windows_htc.shape)
print("subjects_htc :", subjects_htc.shape)

X_windows_htc: (5056, 14, 512)
y_windows_htc: (5056,)
subjects_htc : (5056,)


In [ ]:
FS = 128
bands = [
    ("theta", 4, 8),
    ("alpha", 8, 12),
    ("beta", 12, 30),
]

N_htc, C_htc, T_htc = X_windows_htc.shape

freqs_htc = np.fft.rfftfreq(T_htc, d=1.0/FS)
X_fft_htc = np.fft.rfft(X_windows_htc, axis=-1)
PSD_htc = (np.abs(X_fft_htc) ** 2) / T_htc

bp_list_htc = []

for name, f_lo, f_hi in bands:
    mask = (freqs_htc >= f_lo) & (freqs_htc < f_hi)
    bp = PSD_htc[..., mask].mean(axis=-1)
    bp_list_htc.append(bp)

X_bp_htc = np.stack(bp_list_htc, axis=-1)

# relative normalization (important)
X_bp_htc = X_bp_htc / (X_bp_htc.sum(axis=-1, keepdims=True) + 1e-8)

X_bp_flat_htc = X_bp_htc.reshape(N_htc, -1).astype(np.float32)

print("X_bp_flat_htc:", X_bp_flat_htc.shape)  # (N, 42)

X_bp_flat_htc: (5056, 42)


In [ ]:
from scipy.signal import butter, filtfilt, hilbert

fs = 128
low, high = 4, 7

b, a = butter(4, [low/(fs/2), high/(fs/2)], btype='band')

plv_feats_htc = []

for i in range(N_htc):
    X = X_windows_htc[i]

    X_filt = np.array([filtfilt(b, a, X[ch]) for ch in range(C_htc)])
    analytic = hilbert(X_filt)
    phase = np.angle(analytic)

    plv_mat = np.zeros((C_htc, C_htc))
    for ch1 in range(C_htc):
        for ch2 in range(ch1, C_htc):
            diff = phase[ch1] - phase[ch2]
            plv = np.abs(np.mean(np.exp(1j * diff)))
            plv_mat[ch1, ch2] = plv
            plv_mat[ch2, ch1] = plv

    iu = np.triu_indices(C_htc)
    plv_feats_htc.append(plv_mat[iu])

X_plv_htc = np.array(plv_feats_htc, dtype=np.float32)

print("X_plv_htc:", X_plv_htc.shape)  # (N, 105)

X_plv_htc: (5056, 105)


In [ ]:
X_cov_htc = np.zeros((N_htc, 105), dtype=np.float32)
X_roi_htc = np.zeros((N_htc, 0), dtype=np.float32)

In [ ]:
X_time_t_htc = torch.tensor(X_windows_htc, dtype=torch.float32)
X_bp_t_htc   = torch.tensor(X_bp_flat_htc, dtype=torch.float32)
X_cov_t_htc  = torch.tensor(X_cov_htc, dtype=torch.float32)
X_plv_t_htc  = torch.tensor(X_plv_htc, dtype=torch.float32)
X_roi_t_htc  = torch.tensor(X_roi_htc, dtype=torch.float32)

y_t_htc = torch.tensor(y_windows_htc, dtype=torch.long)

print("X_time_t_htc:", X_time_t_htc.shape)
print("X_bp_t_htc:", X_bp_t_htc.shape)
print("X_plv_t_htc:", X_plv_t_htc.shape)
print("y_t_htc:", y_t_htc.shape)

X_time_t_htc: torch.Size([5056, 14, 512])
X_bp_t_htc: torch.Size([5056, 42])
X_plv_t_htc: torch.Size([5056, 105])
y_t_htc: torch.Size([5056])


In [ ]:
all_subjects_htc = np.unique(subjects_htc)
print("HTC subjects:", all_subjects_htc)

HTC subjects: ['subject_01' 'subject_02' 'subject_03' 'subject_06' 'subject_08'
 'subject_12' 'subject_16' 'subject_17' 'subject_18' 'subject_19'
 'subject_20' 'subject_21' 'subject_22' 'subject_23' 'subject_24'
 'subject_25' 'subject_26']


In [ ]:
from scipy.signal import butter, filtfilt, hilbert

fs = 128

# ----------------------
# THETA (4–7 Hz)
# ----------------------
b_t, a_t = butter(4, [4/(fs/2), 7/(fs/2)], btype='band')

# ----------------------
# ALPHA (8–12 Hz)
# ----------------------
b_a, a_a = butter(4, [8/(fs/2), 12/(fs/2)], btype='band')

plv_theta = []
plv_alpha = []

for i in range(N_htc):
    X = X_windows_htc[i]

    # ---------- THETA ----------
    X_filt_t = np.array([filtfilt(b_t, a_t, X[ch]) for ch in range(C_htc)])
    phase_t = np.angle(hilbert(X_filt_t))

    plv_mat_t = np.zeros((C_htc, C_htc))
    for ch1 in range(C_htc):
        for ch2 in range(ch1, C_htc):
            diff = phase_t[ch1] - phase_t[ch2]
            val = np.abs(np.mean(np.exp(1j * diff)))
            plv_mat_t[ch1, ch2] = val
            plv_mat_t[ch2, ch1] = val

    iu = np.triu_indices(C_htc)
    plv_theta.append(plv_mat_t[iu])

    # ---------- ALPHA ----------
    X_filt_a = np.array([filtfilt(b_a, a_a, X[ch]) for ch in range(C_htc)])
    phase_a = np.angle(hilbert(X_filt_a))

    plv_mat_a = np.zeros((C_htc, C_htc))
    for ch1 in range(C_htc):
        for ch2 in range(ch1, C_htc):
            diff = phase_a[ch1] - phase_a[ch2]
            val = np.abs(np.mean(np.exp(1j * diff)))
            plv_mat_a[ch1, ch2] = val
            plv_mat_a[ch2, ch1] = val

    plv_alpha.append(plv_mat_a[iu])

# combine
X_plv_htc = np.concatenate([plv_theta, plv_alpha], axis=1).astype(np.float32)

print("X_plv_htc:", X_plv_htc.shape)  # should be (N, 210)

X_plv_htc: (5056, 210)


In [ ]:
X_plv_t_htc = torch.tensor(X_plv_htc, dtype=torch.float32)

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10  # ↓ reduced to avoid overfitting
BATCH  = 396
LR     = 5e-4
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.7

overall_results_htc = []

for fold_i, test_subj in enumerate(all_subjects_htc):

    print("\n" + "="*80)
    print(f"HTC FOLD {fold_i+1}/{len(all_subjects_htc)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects_htc == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_time_train = X_time_t_htc[train_idx].clone()
    X_time_test  = X_time_t_htc[test_idx].clone()

    X_bp_train = X_bp_t_htc[train_idx].clone()
    X_bp_test  = X_bp_t_htc[test_idx].clone()

    # 🔥 COVARIANCE ENABLED
    X_cov_train = X_cov_t_htc[train_idx].clone()
    X_cov_test  = X_cov_t_htc[test_idx].clone()

    X_plv_train = X_plv_t_htc[train_idx].clone()
    X_plv_test  = X_plv_t_htc[test_idx].clone()

    y_train = y_t_htc[train_idx].clone()
    y_test  = y_t_htc[test_idx].clone()

    subj_train = subjects_htc[train_idx]
    subj_test  = subjects_htc[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================

    # TIME
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # BANDPOWER
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # 🔥 COV NORMALIZATION (NEW)
    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    # ROI disabled (consistent with your setup)
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train,
        X_plv_train, X_roi_train, y_train, subj_train
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test,
        X_plv_test, X_roi_test, y_test, subj_test
    )

    sampler = SubjectBalancedBatchSampler(
        train_ds, batch_size=BATCH, subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():

            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0)
                for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)

    # =========================
    # FEW-SHOT (UNCHANGED)
    # =========================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("HTC AdaBN + Mahalanobis few-shot acc:", fewshot_acc)

    overall_results_htc.append((test_subj, fewshot_acc))


print("\nHTC FINAL SUMMARY")
accs = [a for _, a in overall_results_htc]
for s, a in overall_results_htc:
    print(s, ":", a)

print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


HTC FOLD 1/17 | TEST SUBJECT = subject_01


NameError: name 'EEGDataset' is not defined

In [ ]:
HTC_ID = 0
NBACK_ID = 1

In [ ]:
subjects_nb_prefixed = np.array([f"nb_{s}" for s in subjects])
print("Example N-back subjects:", subjects_nb_prefixed[:5])

Example N-back subjects: ['nb_subject_01' 'nb_subject_01' 'nb_subject_01' 'nb_subject_01'
 'nb_subject_01']


In [ ]:
subjects_htc_prefixed = np.array([f"htc_{s}" for s in subjects_htc])
print("Example HTC subjects:", subjects_htc_prefixed[:5])

Example HTC subjects: ['htc_subject_01' 'htc_subject_01' 'htc_subject_01' 'htc_subject_01'
 'htc_subject_01']


In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset

class EEGDataset(Dataset):
    def __init__(self, X_time, X_bp, X_cov, X_plv, X_roi, y, subj_strs, dataset_id):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_cov  = X_cov
        self.X_plv  = X_plv
        self.X_roi  = X_roi
        self.y      = y

        self.dataset_id = int(dataset_id)

        # map subject strings to local integer IDs for this dataset object
        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_cov[idx],
            self.X_plv[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx],
            torch.tensor(self.dataset_id, dtype=torch.long)
        )

In [ ]:
def extract_embeddings(model, loader, device):

    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        # for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in loader:
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    E = torch.cat(E, dim=0)
    Y = torch.cat(Y, dim=0)

    return E, Y

In [ ]:
class EEGDataset(torch.utils.data.Dataset):
    def __init__(self, X_time, X_bp, X_cov, X_plv, X_roi, y, subj_strs, dataset_id):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_cov  = X_cov
        self.X_plv  = X_plv
        self.X_roi  = X_roi
        self.y      = y

        self.dataset_id = int(dataset_id)

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_cov[idx],
            self.X_plv[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx],
            torch.tensor(self.dataset_id, dtype=torch.long)
        )

In [ ]:
for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
    print("Batch size:", len(yb))
    print("Dataset IDs:", torch.unique(db))
    break

NameError: name 'train_loader' is not defined

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    # X_cov_train = X_cov_t[train_idx].clone()
    # X_cov_test  = X_cov_t[test_idx].clone()

    # cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    # cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    # X_cov_train = (X_cov_train - cov_mu) / cov_sd
    # X_cov_test  = (X_cov_test  - cov_mu) / cov_sd
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # # =========================
    # X_roi_train = build_roi_workload_features_from_bp(
    #     X_bp_train,
    #     # frontal_idx=(0,1,2,3),
    #     frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )
    # X_roi_test = build_roi_workload_features_from_bp(
    #     X_bp_test,
    #     frontal_idx=(0,1,2,3),
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )

    # roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    # roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    # X_roi_train = (X_roi_train - roi_mu) / roi_sd
    # X_roi_test  = (X_roi_test  - roi_mu) / roi_sd
    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    # train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    # test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)
    train_ds = EEGDataset(
      X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train,
      y_train,
      subjects_nb_prefixed[train_idx],   # 🔥 use prefixed subjects
      dataset_id=NBACK_ID                # 🔥 NEW
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test, X_plv_test, X_roi_test,
        y_test,
        subjects_nb_prefixed[test_idx],    # 🔥 use prefixed subjects
        dataset_id=NBACK_ID                # 🔥 NEW
    )

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        # for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        # for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # =====================================
    # FEATURE RELIABILITY WEIGHTING
    # =====================================
    # compute variance of each embedding dimension
    var = e_support.var(dim=0)
    # reliability = inverse variance
    weights = 1.0 / (var + 1e-6)
    # normalize weights
    weights = weights / weights.mean()
    # apply weighting
    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6078 | best 0.6078
Epoch 02 | zero-shot acc 0.6667 | best 0.6667
Epoch 03 | zero-shot acc 0.7045 | best 0.7045
Epoch 04 | zero-shot acc 0.7161 | best 0.7161
Epoch 05 | zero-shot acc 0.6951 | best 0.7161
Epoch 06 | zero-shot acc 0.6909 | best 0.7161
Epoch 07 | zero-shot acc 0.6898 | best 0.7161
Epoch 08 | zero-shot acc 0.7161 | best 0.7161
Epoch 09 | zero-shot acc 0.7066 | best 0.7161
Epoch 10 | zero-shot acc 0.7298 | best 0.7298
AdaBN + Mahalanobis few-shot acc: 0.790123462677002

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4920 | best 0.4920
Epoch 02 | zero-shot acc 0.5474 | best 0.5474
Epoch 03 | zero-shot acc 0.5602 | best 0.5602
Epoch 04 | zero-shot acc 0.4963 | best 0.5602
Epoch 05 | zero-shot acc 0.6070 | best 0.6070
Epoch 06 | zero-shot acc 0.5793 | best 0.6070
Epoch 07 | zero-shot acc 0.5442 | best 0.6070
Epoch 08 | zero-shot acc 0.6177 | best 0.6177
Epoch 09 | zero-shot acc 0.5985 | best 0.6

In [ ]:
# =========================
# LOAD N-BACK
# =========================
data_nb = torch.load(
    "/content/drive/MyDrive/eeg_preprocessed_nback.pt",
    weights_only=False
)

X_time_t = data_nb["X_time"]
X_bp_t   = data_nb["X_bp"]
X_cov_t  = data_nb["X_cov"]
X_plv_t  = data_nb["X_plv"]
y_t      = data_nb["y"]
subjects = data_nb["subjects"]

print("Loaded N-back")


# =========================
# LOAD HTC
# =========================
data_htc = torch.load(
    "/content/drive/MyDrive/eeg_preprocessed_htc.pt",
    weights_only=False
)

X_time_t_htc = data_htc["X_time"]
X_bp_t_htc   = data_htc["X_bp"]
X_cov_t_htc  = data_htc["X_cov"]
X_plv_t_htc  = data_htc["X_plv"]
y_t_htc      = data_htc["y"]
subjects_htc = data_htc["subjects"]

print("Loaded HTC")

Loaded N-back
Loaded HTC


In [ ]:
print("\n--- N-BACK ---")
print(X_time_t.shape, X_bp_t.shape, X_cov_t.shape, X_plv_t.shape, y_t.shape)

print("\n--- HTC ---")
print(X_time_t_htc.shape, X_bp_t_htc.shape, X_cov_t_htc.shape, X_plv_t_htc.shape, y_t_htc.shape)


--- N-BACK ---
torch.Size([12159, 14, 512]) torch.Size([12159, 42]) torch.Size([15007, 105]) torch.Size([15007, 210]) torch.Size([12159])

--- HTC ---
torch.Size([5056, 14, 512]) torch.Size([5056, 42]) torch.Size([5056, 105]) torch.Size([5056, 210]) torch.Size([5056])


In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 5e-4
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    # X_cov_train = X_cov_t[train_idx].clone()
    # X_cov_test  = X_cov_t[test_idx].clone()

    # cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    # cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    # X_cov_train = (X_cov_train - cov_mu) / cov_sd
    # X_cov_test  = (X_cov_test  - cov_mu) / cov_sd
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # # =========================
    # X_roi_train = build_roi_workload_features_from_bp(
    #     X_bp_train,
    #     # frontal_idx=(0,1,2,3),
    #     frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )
    # X_roi_test = build_roi_workload_features_from_bp(
    #     X_bp_test,
    #     frontal_idx=(0,1,2,3),
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )

    # roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    # roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    # X_roi_train = (X_roi_train - roi_mu) / roi_sd
    # X_roi_test  = (X_roi_test  - roi_mu) / roi_sd
    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    # train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    # test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)
    train_ds = EEGDataset(
      X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train,
      y_train,
      subjects_nb_prefixed[train_idx],   # 🔥 use prefixed subjects
      dataset_id=NBACK_ID                # 🔥 NEW
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test, X_plv_test, X_roi_test,
        y_test,
        subjects_nb_prefixed[test_idx],    # 🔥 use prefixed subjects
        dataset_id=NBACK_ID                # 🔥 NEW
    )

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        # for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        # for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # =====================================
    # FEATURE RELIABILITY WEIGHTING
    # =====================================
    # compute variance of each embedding dimension
    var = e_support.var(dim=0)
    # reliability = inverse variance
    weights = 1.0 / (var + 1e-6)
    # normalize weights
    weights = weights / weights.mean()
    # apply weighting
    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6299 | best 0.6299
Epoch 02 | zero-shot acc 0.6688 | best 0.6688
Epoch 03 | zero-shot acc 0.6793 | best 0.6793
Epoch 04 | zero-shot acc 0.7098 | best 0.7098
Epoch 05 | zero-shot acc 0.7171 | best 0.7171
Epoch 06 | zero-shot acc 0.7119 | best 0.7171
Epoch 07 | zero-shot acc 0.7340 | best 0.7340
Epoch 08 | zero-shot acc 0.7329 | best 0.7340
Epoch 09 | zero-shot acc 0.7392 | best 0.7392
Epoch 10 | zero-shot acc 0.7213 | best 0.7392
AdaBN + Mahalanobis few-shot acc: 0.824915885925293

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4302 | best 0.4302
Epoch 02 | zero-shot acc 0.4665 | best 0.4665
Epoch 03 | zero-shot acc 0.4941 | best 0.4941
Epoch 04 | zero-shot acc 0.5144 | best 0.5144
Epoch 05 | zero-shot acc 0.5410 | best 0.5410
Epoch 06 | zero-shot acc 0.5495 | best 0.5495
Epoch 07 | zero-shot acc 0.5208 | best 0.5495
Epoch 08 | zero-shot acc 0.5325 | best 0.5495
Epoch 09 | zero-shot acc 0.5740 | best 0.5

In [ ]:
'''removed bn adapt here'''

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects_nb_prefixed[train_idx]
    subj_test  = subjects_nb_prefixed[test_idx]

    # =========================
    # NORMALIZATION
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train,
        y_train,
        subj_train,
        dataset_id=NBACK_ID
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test, X_plv_test, X_roi_test,
        y_test,
        subj_test,
        dataset_id=NBACK_ID
    )

    sampler = SubjectBalancedBatchSampler(
        train_ds, batch_size=BATCH, subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():

            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0)
                for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # =========================
    # LOAD BEST MODEL
    # =========================
    model.load_state_dict(best_state)

    # =========================
    # FEW-SHOT (NO BN ADAPT)
    # =========================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # normalize
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # feature weighting
    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("N-back Mahalanobis few-shot acc (NO BN):", fewshot_acc)

    overall_results.append((test_subj, fewshot_acc))


print("\nN-BACK FINAL SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)

print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6193 | best 0.6193
Epoch 02 | zero-shot acc 0.6614 | best 0.6614
Epoch 03 | zero-shot acc 0.6835 | best 0.6835
Epoch 04 | zero-shot acc 0.6772 | best 0.6835
Epoch 05 | zero-shot acc 0.7066 | best 0.7066
Epoch 06 | zero-shot acc 0.7234 | best 0.7234
Epoch 07 | zero-shot acc 0.6972 | best 0.7234
Epoch 08 | zero-shot acc 0.7182 | best 0.7234
Epoch 09 | zero-shot acc 0.7161 | best 0.7234
Epoch 10 | zero-shot acc 0.7234 | best 0.7234
N-back Mahalanobis few-shot acc (NO BN): 0.8518518805503845

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4175 | best 0.4175
Epoch 02 | zero-shot acc 0.5261 | best 0.5261
Epoch 03 | zero-shot acc 0.5442 | best 0.5442
Epoch 04 | zero-shot acc 0.5900 | best 0.5900
Epoch 05 | zero-shot acc 0.5293 | best 0.5900
Epoch 06 | zero-shot acc 0.5325 | best 0.5900
Epoch 07 | zero-shot acc 0.5676 | best 0.5900
Epoch 08 | zero-shot acc 0.5793 | best 0.5900
Epoch 09 | zero-shot acc 0.5517 | 

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4
SHIFT_THRESH = 0.14

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURES
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects_nb_prefixed[train_idx]
    subj_test  = subjects_nb_prefixed[test_idx]

    # =========================
    # NORMALIZE
    # =========================
    time_mu = X_time_train.mean((0,2), keepdim=True)
    time_sd = X_time_train.std((0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(0, keepdim=True)
    bp_sd = X_bp_train.std(0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train,
        y_train, subj_train, dataset_id=NBACK_ID
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test, X_plv_test, X_roi_test,
        y_test, subj_test, dataset_id=NBACK_ID
    )

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(64,64,42,0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss = (
                prototype_ce_loss(e, yb, 0.1)
                + LAMBDA * supcon_diff_subject_loss(proj, yb, sb, TEMP)
                + PROTO_LAMBDA * prototype_loss(e, yb)
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():

            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            # SHIFT (for gating)
            shift = torch.norm(e_train.mean(0) - e_test.mean(0)).item()

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0)
                for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f} | shift {shift:.4f}")

    # =========================
    # LOAD BEST MODEL
    # =========================
    model.load_state_dict(best_state)

    # =========================
    # FEW-SHOT PREP
    # =========================
    all_e, all_y = extract_embeddings(model, DataLoader(test_ds,256), device)
    all_e = all_e.to(device)
    all_y = all_y.to(device)

    support_idx = torch.cat([
        (all_y==c).nonzero(as_tuple=True)[0][torch.randperm((all_y==c).sum())[:FEW_SHOT_PER_CLASS]]
        for c in torch.unique(all_y)
    ])

    mask = torch.zeros(len(all_y), dtype=torch.bool, device=device)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.cpu())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    # =========================
    # BN ADAPT (GATED)
    # =========================
    use_bn = shift > SHIFT_THRESH
    print(f"BN USED: {use_bn}")

    if use_bn:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract
        all_e, all_y = extract_embeddings(model, DataLoader(test_ds,256), device)
        all_e = all_e.to(device)
        all_y = all_y.to(device)

    # =========================
    # MAHALANOBIS FEW-SHOT
    # =========================
    e_support = all_e[mask]
    y_support = all_y[mask]
    e_query   = all_e[~mask]
    y_query   = all_y[~mask]

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(0)
    w = (1/(var+1e-6)); w/=w.mean()

    e_support *= w
    e_query   *= w

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support==c].mean(0) for c in classes])
    protos = F.normalize(protos,dim=1)

    res = torch.cat([e_support[y_support==c]-protos[i] for i,c in enumerate(classes)],0)

    D = res.shape[1]
    cov = (res.t()@res)/max(res.shape[0]-1,1)
    cov = cov.to(device)

    avg = torch.trace(cov)/D
    cov = (1-SHRINK)*cov + SHRINK*(avg*torch.eye(D,device=device))
    cov_inv = torch.linalg.pinv(cov)

    diffs = e_query.unsqueeze(1)-protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk",diffs,cov_inv,diffs)

    fewshot_acc = (dists.argmin(1)==y_query).float().mean().item()

    print(f"FEW-SHOT ACC: {fewshot_acc:.4f}")

    overall_results.append((test_subj, fewshot_acc))


# =========================
# FINAL SUMMARY
# =========================
print("\n" + "="*50)
print("N-BACK FINAL SUMMARY")
print("="*50)

accs = [a for _, a in overall_results]

for s, a in overall_results:
    print(f"{s:12s} : {a*100:.2f}%")

print("-"*50)
print(f"Mean accuracy : {np.mean(accs)*100:.2f}%")
print(f"Std  dev      : {np.std(accs)*100:.2f}%")
print("="*50)


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6046 | best 0.6046 | shift 0.1756
Epoch 02 | zero-shot acc 0.7045 | best 0.7045 | shift 0.1630
Epoch 03 | zero-shot acc 0.7234 | best 0.7234 | shift 0.1464
Epoch 04 | zero-shot acc 0.7077 | best 0.7234 | shift 0.1459
Epoch 05 | zero-shot acc 0.7192 | best 0.7234 | shift 0.1222
Epoch 06 | zero-shot acc 0.7129 | best 0.7234 | shift 0.1350
Epoch 07 | zero-shot acc 0.6940 | best 0.7234 | shift 0.1616
Epoch 08 | zero-shot acc 0.7277 | best 0.7277 | shift 0.1313
Epoch 09 | zero-shot acc 0.7234 | best 0.7277 | shift 0.1408
Epoch 10 | zero-shot acc 0.7056 | best 0.7277 | shift 0.1133
BN USED: False
FEW-SHOT ACC: 0.8126

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.3951 | best 0.3951 | shift 0.4127
Epoch 02 | zero-shot acc 0.4590 | best 0.4590 | shift 0.3644
Epoch 03 | zero-shot acc 0.5474 | best 0.5474 | shift 0.2608
Epoch 04 | zero-shot acc 0.5325 | best 0.5474 | shift 0.1926
Epoch 05 | zero-shot acc 0.4452 

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 5e-4
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.7
SHIFT_THRESH = 0.15   # 🔥 slightly higher for HTC

overall_results_htc = []

for fold_i, test_subj in enumerate(all_subjects_htc):

    print("\n" + "="*80)
    print(f"HTC FOLD {fold_i+1}/{len(all_subjects_htc)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects_htc == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURES
    # =========================
    X_time_train = X_time_t_htc[train_idx].clone()
    X_time_test  = X_time_t_htc[test_idx].clone()

    X_bp_train = X_bp_t_htc[train_idx].clone()
    X_bp_test  = X_bp_t_htc[test_idx].clone()

    X_cov_train = X_cov_t_htc[train_idx].clone()
    X_cov_test  = X_cov_t_htc[test_idx].clone()

    X_plv_train = X_plv_t_htc[train_idx].clone()
    X_plv_test  = X_plv_t_htc[test_idx].clone()

    y_train = y_t_htc[train_idx].clone()
    y_test  = y_t_htc[test_idx].clone()

    subj_train = subjects_htc_prefixed[train_idx]
    subj_test  = subjects_htc_prefixed[test_idx]

    # =========================
    # NORMALIZATION
    # =========================
    time_mu = X_time_train.mean((0,2), keepdim=True)
    time_sd = X_time_train.std((0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(0, keepdim=True)
    bp_sd = X_bp_train.std(0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    cov_mu = X_cov_train.mean(0, keepdim=True)
    cov_sd = X_cov_train.std(0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    X_roi_train = torch.zeros((len(train_idx), 0))
    X_roi_test  = torch.zeros((len(test_idx), 0))

    # =========================
    # DATASET
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train,
        X_plv_train, X_roi_train,
        y_train, subj_train,
        dataset_id=HTC_ID
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test,
        X_plv_test, X_roi_test,
        y_test, subj_test,
        dataset_id=HTC_ID
    )

    sampler = SubjectBalancedBatchSampler(
        train_ds, batch_size=BATCH, subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(64,64,42,0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss = (
                prototype_ce_loss(e, yb, 0.1)
                + LAMBDA * supcon_diff_subject_loss(proj, yb, sb, TEMP)
                + PROTO_LAMBDA * prototype_loss(e, yb)
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT
        # =========================
        model.eval()
        with torch.no_grad():

            train_eval_loader = DataLoader(train_ds,256)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            shift = torch.norm(e_train.mean(0) - e_test.mean(0)).item()

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0)
                for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred = logits.argmax(1)

            label_map = {int(c.item()): i for i,c in enumerate(classes)}
            y_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred.device
            )

            acc = (pred == y_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f} | shift {shift:.4f}")

    # =========================
    # LOAD BEST MODEL
    # =========================
    model.load_state_dict(best_state)

    # =========================
    # FEW-SHOT PREP
    # =========================
    all_e, all_y = extract_embeddings(model, DataLoader(test_ds,256), device)
    all_e = all_e.to(device)
    all_y = all_y.to(device)

    support_idx = torch.cat([
        (all_y==c).nonzero(as_tuple=True)[0][torch.randperm((all_y==c).sum())[:FEW_SHOT_PER_CLASS]]
        for c in torch.unique(all_y)
    ])

    mask = torch.zeros(len(all_y), dtype=torch.bool, device=device)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.cpu())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    # =========================
    # BN ADAPT (GATED)
    # =========================
    use_bn = shift > SHIFT_THRESH
    print(f"BN USED: {use_bn}")

    if use_bn:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract
        all_e, all_y = extract_embeddings(model, DataLoader(test_ds,256), device)
        all_e = all_e.to(device)
        all_y = all_y.to(device)

    # =========================
    # MAHALANOBIS
    # =========================
    e_support = all_e[mask]
    y_support = all_y[mask]
    e_query   = all_e[~mask]
    y_query   = all_y[~mask]

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(0)
    w = (1/(var+1e-6)); w/=w.mean()

    e_support *= w
    e_query   *= w

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support==c].mean(0) for c in classes])
    protos = F.normalize(protos,dim=1)

    res = torch.cat([e_support[y_support==c]-protos[i] for i,c in enumerate(classes)],0)

    D = res.shape[1]
    cov = (res.t()@res)/max(res.shape[0]-1,1)
    cov = cov.to(device)

    avg = torch.trace(cov)/D
    cov = (1-SHRINK)*cov + SHRINK*(avg*torch.eye(D,device=device))
    cov_inv = torch.linalg.pinv(cov)

    diffs = e_query.unsqueeze(1)-protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk",diffs,cov_inv,diffs)

    fewshot_acc = (dists.argmin(1)==y_query).float().mean().item()
    print(f"FEW-SHOT ACC: {fewshot_acc:.4f}")

    overall_results_htc.append((test_subj, fewshot_acc))


# =========================
# FINAL SUMMARY
# =========================
print("\n" + "="*50)
print("HTC FINAL SUMMARY")
print("="*50)

accs = [a for _, a in overall_results_htc]

for s, a in overall_results_htc:
    print(f"{s:12s} : {a*100:.2f}%")

print("-"*50)
print(f"Mean accuracy : {np.mean(accs)*100:.2f}%")
print(f"Std  dev      : {np.std(accs)*100:.2f}%")
print("="*50)


HTC FOLD 1/17 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5700 | best 0.5700 | shift 0.0913
Epoch 02 | zero-shot acc 0.5765 | best 0.5765 | shift 0.1458
Epoch 03 | zero-shot acc 0.5863 | best 0.5863 | shift 0.4580
Epoch 04 | zero-shot acc 0.5635 | best 0.5863 | shift 1.1598
Epoch 05 | zero-shot acc 0.6189 | best 0.6189 | shift 1.9199
Epoch 06 | zero-shot acc 0.6287 | best 0.6287 | shift 2.1038
Epoch 07 | zero-shot acc 0.6384 | best 0.6384 | shift 1.7443
Epoch 08 | zero-shot acc 0.6319 | best 0.6384 | shift 1.5042
Epoch 09 | zero-shot acc 0.6417 | best 0.6417 | shift 1.1351
Epoch 10 | zero-shot acc 0.6417 | best 0.6417 | shift 0.9711
BN USED: True
FEW-SHOT ACC: 0.7453

HTC FOLD 2/17 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4983 | best 0.4983 | shift 0.0626
Epoch 02 | zero-shot acc 0.5518 | best 0.5518 | shift 0.1122
Epoch 03 | zero-shot acc 0.7258 | best 0.7258 | shift 0.1982
Epoch 04 | zero-shot acc 0.6990 | best 0.7258 | shift 0.3253
Epoch 05 | zero-shot acc 

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10  # ↓ reduced to avoid overfitting
BATCH  = 396
LR     = 5e-4
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.7

overall_results_htc = []

for fold_i, test_subj in enumerate(all_subjects_htc):

    print("\n" + "="*80)
    print(f"HTC FOLD {fold_i+1}/{len(all_subjects_htc)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects_htc == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_time_train = X_time_t_htc[train_idx].clone()
    X_time_test  = X_time_t_htc[test_idx].clone()

    X_bp_train = X_bp_t_htc[train_idx].clone()
    X_bp_test  = X_bp_t_htc[test_idx].clone()

    # 🔥 COVARIANCE ENABLED
    X_cov_train = X_cov_t_htc[train_idx].clone()
    X_cov_test  = X_cov_t_htc[test_idx].clone()

    X_plv_train = X_plv_t_htc[train_idx].clone()
    X_plv_test  = X_plv_t_htc[test_idx].clone()

    y_train = y_t_htc[train_idx].clone()
    y_test  = y_t_htc[test_idx].clone()

    subj_train = subjects_htc[train_idx]
    subj_test  = subjects_htc[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================

    # TIME
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # BANDPOWER
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # 🔥 COV NORMALIZATION (NEW)
    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    # ROI disabled (consistent with your setup)
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    # train_ds = EEGDataset(
    #     X_time_train, X_bp_train, X_cov_train,
    #     X_plv_train, X_roi_train, y_train, subj_train
    # )

    # test_ds = EEGDataset(
    #     X_time_test, X_bp_test, X_cov_test,
    #     X_plv_test, X_roi_test, y_test, subj_test
    # )
    train_ds = EEGDataset(
      X_time_train, X_bp_train, X_cov_train,
      X_plv_train, X_roi_train,
      y_train,
      subjects_htc_prefixed[train_idx],   # 🔥 prefixed
      dataset_id=HTC_ID                  # 🔥 NEW
)
    test_ds = EEGDataset(
      X_time_test, X_bp_test, X_cov_test,
      X_plv_test, X_roi_test,
      y_test,
      subjects_htc_prefixed[test_idx],   # 🔥 prefixed
      dataset_id=HTC_ID
)

    sampler = SubjectBalancedBatchSampler(
        train_ds, batch_size=BATCH, subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        # for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
        # for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
        # for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader:
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():

            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0)
                for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)

    # =========================
    # FEW-SHOT (UNCHANGED)
    # =========================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("HTC AdaBN + Mahalanobis few-shot acc:", fewshot_acc)

    overall_results_htc.append((test_subj, fewshot_acc))


print("\nHTC FINAL SUMMARY")
accs = [a for _, a in overall_results_htc]
for s, a in overall_results_htc:
    print(s, ":", a)

print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


HTC FOLD 1/17 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.4169 | best 0.4169
Epoch 02 | zero-shot acc 0.5375 | best 0.5375
Epoch 03 | zero-shot acc 0.6319 | best 0.6319
Epoch 04 | zero-shot acc 0.6417 | best 0.6417
Epoch 05 | zero-shot acc 0.6254 | best 0.6417
Epoch 06 | zero-shot acc 0.6091 | best 0.6417
Epoch 07 | zero-shot acc 0.6384 | best 0.6417
Epoch 08 | zero-shot acc 0.6026 | best 0.6417
Epoch 09 | zero-shot acc 0.5668 | best 0.6417
Epoch 10 | zero-shot acc 0.5407 | best 0.6417
HTC AdaBN + Mahalanobis few-shot acc: 0.7415730357170105

HTC FOLD 2/17 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.5151 | best 0.5151
Epoch 02 | zero-shot acc 0.5117 | best 0.5151
Epoch 03 | zero-shot acc 0.6154 | best 0.6154
Epoch 04 | zero-shot acc 0.7124 | best 0.7124
Epoch 05 | zero-shot acc 0.6722 | best 0.7124
Epoch 06 | zero-shot acc 0.5585 | best 0.7124
Epoch 07 | zero-shot acc 0.5151 | best 0.7124
Epoch 08 | zero-shot acc 0.5485 | best 0.7124
Epoch 09 | zero-shot acc 0.55

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10  # ↓ reduced to avoid overfitting
BATCH  = 396
LR     = 5e-4
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.7

overall_results_htc = []

for fold_i, test_subj in enumerate(all_subjects_htc):

    print("\n" + "="*80)
    print(f"HTC FOLD {fold_i+1}/{len(all_subjects_htc)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects_htc == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_time_train = X_time_t_htc[train_idx].clone()
    X_time_test  = X_time_t_htc[test_idx].clone()

    X_bp_train = X_bp_t_htc[train_idx].clone()
    X_bp_test  = X_bp_t_htc[test_idx].clone()

    # 🔥 COVARIANCE ENABLED
    X_cov_train = X_cov_t_htc[train_idx].clone()
    X_cov_test  = X_cov_t_htc[test_idx].clone()

    X_plv_train = X_plv_t_htc[train_idx].clone()
    X_plv_test  = X_plv_t_htc[test_idx].clone()

    y_train = y_t_htc[train_idx].clone()
    y_test  = y_t_htc[test_idx].clone()

    subj_train = subjects_htc[train_idx]
    subj_test  = subjects_htc[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================

    # TIME
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # BANDPOWER
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # 🔥 COV NORMALIZATION (NEW)
    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    # ROI disabled (consistent with your setup)
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    # train_ds = EEGDataset(
    #     X_time_train, X_bp_train, X_cov_train,
    #     X_plv_train, X_roi_train, y_train, subj_train
    # )

    # test_ds = EEGDataset(
    #     X_time_test, X_bp_test, X_cov_test,
    #     X_plv_test, X_roi_test, y_test, subj_test
    # )
    train_ds = EEGDataset(
      X_time_train, X_bp_train, X_cov_train,
      X_plv_train, X_roi_train,
      y_train,
      subjects_htc_prefixed[train_idx],   # 🔥 prefixed
      dataset_id=HTC_ID                  # 🔥 NEW
)
    test_ds = EEGDataset(
      X_time_test, X_bp_test, X_cov_test,
      X_plv_test, X_roi_test,
      y_test,
      subjects_htc_prefixed[test_idx],   # 🔥 prefixed
      dataset_id=HTC_ID
)

    sampler = SubjectBalancedBatchSampler(
        train_ds, batch_size=BATCH, subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        # for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
        # for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
        # for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader:
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():

            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0)
                for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)

    # =========================
    # FEW-SHOT (UNCHANGED)
    # =========================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("HTC AdaBN + Mahalanobis few-shot acc:", fewshot_acc)

    overall_results_htc.append((test_subj, fewshot_acc))


print("\nHTC FINAL SUMMARY")
accs = [a for _, a in overall_results_htc]
for s, a in overall_results_htc:
    print(s, ":", a)

print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


HTC FOLD 1/17 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5700 | best 0.5700
Epoch 02 | zero-shot acc 0.5928 | best 0.5928
Epoch 03 | zero-shot acc 0.6743 | best 0.6743
Epoch 04 | zero-shot acc 0.6971 | best 0.6971
Epoch 05 | zero-shot acc 0.6906 | best 0.6971
Epoch 06 | zero-shot acc 0.6645 | best 0.6971
Epoch 07 | zero-shot acc 0.6775 | best 0.6971
Epoch 08 | zero-shot acc 0.6775 | best 0.6971
Epoch 09 | zero-shot acc 0.6352 | best 0.6971
Epoch 10 | zero-shot acc 0.6254 | best 0.6971
HTC AdaBN + Mahalanobis few-shot acc: 0.7228464484214783

HTC FOLD 2/17 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.5686 | best 0.5686
Epoch 02 | zero-shot acc 0.6388 | best 0.6388
Epoch 03 | zero-shot acc 0.6656 | best 0.6656
Epoch 04 | zero-shot acc 0.7224 | best 0.7224
Epoch 05 | zero-shot acc 0.6890 | best 0.7224
Epoch 06 | zero-shot acc 0.5017 | best 0.7224
Epoch 07 | zero-shot acc 0.5920 | best 0.7224
Epoch 08 | zero-shot acc 0.5652 | best 0.7224
Epoch 09 | zero-shot acc 0.51

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 5e-4
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.7

overall_results_htc = []

for fold_i, test_subj in enumerate(all_subjects_htc):

    print("\n" + "="*80)
    print(f"HTC FOLD {fold_i+1}/{len(all_subjects_htc)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects_htc == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_time_train = X_time_t_htc[train_idx].clone()
    X_time_test  = X_time_t_htc[test_idx].clone()

    X_bp_train = X_bp_t_htc[train_idx].clone()
    X_bp_test  = X_bp_t_htc[test_idx].clone()

    X_cov_train = X_cov_t_htc[train_idx].clone()
    X_cov_test  = X_cov_t_htc[test_idx].clone()

    X_plv_train = X_plv_t_htc[train_idx].clone()
    X_plv_test  = X_plv_t_htc[test_idx].clone()

    y_train = y_t_htc[train_idx].clone()
    y_test  = y_t_htc[test_idx].clone()

    subj_train = subjects_htc[train_idx]
    subj_test  = subjects_htc[test_idx]

    # =========================
    # NORMALIZATION
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train,
        X_plv_train, X_roi_train,
        y_train,
        subj_train,
        dataset_id=HTC_ID   # 🔥 FIX
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test,
        X_plv_test, X_roi_test,
        y_test,
        subj_test,
        dataset_id=HTC_ID   # 🔥 FIX
    )

    sampler = SubjectBalancedBatchSampler(
        train_ds, batch_size=BATCH, subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():

            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0)
                for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # =========================
    # LOAD BEST MODEL
    # =========================
    model.load_state_dict(best_state)

    # =========================
    # FEW-SHOT + PARTIAL BN ADAPT
    # =========================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    # 🔥 PARTIAL BN ADAPT (IDENTICAL TO N-BACK)
    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    # =========================
    # RE-EXTRACT EMBEDDINGS
    # =========================
    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("HTC AdaBN + Mahalanobis few-shot acc:", fewshot_acc)

    overall_results_htc.append((test_subj, fewshot_acc))


print("\nHTC FINAL SUMMARY")
accs = [a for _, a in overall_results_htc]
for s, a in overall_results_htc:
    print(s, ":", a)

print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


HTC FOLD 1/17 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5147 | best 0.5147
Epoch 02 | zero-shot acc 0.5375 | best 0.5375
Epoch 03 | zero-shot acc 0.6840 | best 0.6840
Epoch 04 | zero-shot acc 0.7622 | best 0.7622
Epoch 05 | zero-shot acc 0.7296 | best 0.7622
Epoch 06 | zero-shot acc 0.6906 | best 0.7622
Epoch 07 | zero-shot acc 0.6938 | best 0.7622
Epoch 08 | zero-shot acc 0.7362 | best 0.7622
Epoch 09 | zero-shot acc 0.6319 | best 0.7622
Epoch 10 | zero-shot acc 0.6156 | best 0.7622
HTC AdaBN + Mahalanobis few-shot acc: 0.7265917658805847

HTC FOLD 2/17 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.5753 | best 0.5753
Epoch 02 | zero-shot acc 0.6555 | best 0.6555
Epoch 03 | zero-shot acc 0.7525 | best 0.7525
Epoch 04 | zero-shot acc 0.7090 | best 0.7525
Epoch 05 | zero-shot acc 0.6355 | best 0.7525
Epoch 06 | zero-shot acc 0.6054 | best 0.7525
Epoch 07 | zero-shot acc 0.5318 | best 0.7525
Epoch 08 | zero-shot acc 0.5987 | best 0.7525
Epoch 09 | zero-shot acc 0.59

In [ ]:

from torch.utils.data import DataLoader, ConcatDataset

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

N_FOLDS = min(len(all_subjects), len(all_subjects_htc))

overall_results_joint = []

for fold_i in range(N_FOLDS):

    test_subj_nb  = all_subjects[fold_i]
    test_subj_htc = all_subjects_htc[fold_i]

    print("\n" + "="*90)
    print(f"JOINT FOLD {fold_i+1}/{N_FOLDS}")
    print(f"N-back test subject: {test_subj_nb}")
    print(f"HTC test subject:    {test_subj_htc}")
    print("="*90)

    # =========================
    # SPLITS
    # =========================
    test_mask_nb  = (subjects == test_subj_nb)
    train_mask_nb = ~test_mask_nb
    train_idx_nb = np.where(train_mask_nb)[0]
    test_idx_nb  = np.where(test_mask_nb)[0]

    test_mask_htc  = (subjects_htc == test_subj_htc)
    train_mask_htc = ~test_mask_htc
    train_idx_htc = np.where(train_mask_htc)[0]
    test_idx_htc  = np.where(test_mask_htc)[0]

    # =========================
    # N-BACK
    # =========================
    X_cov_train_nb = torch.zeros_like(X_cov_t[train_idx_nb])
    X_cov_test_nb  = torch.zeros_like(X_cov_t[test_idx_nb])

    X_plv_train_nb = X_plv_t[train_idx_nb].clone()
    X_plv_test_nb  = X_plv_t[test_idx_nb].clone()

    X_time_train_nb = X_time_t[train_idx_nb].clone()
    X_time_test_nb  = X_time_t[test_idx_nb].clone()

    X_bp_train_nb = X_bp_t[train_idx_nb].clone()
    X_bp_test_nb  = X_bp_t[test_idx_nb].clone()

    y_train_nb = y_t[train_idx_nb].clone()
    y_test_nb  = y_t[test_idx_nb].clone()

    time_mu_nb = X_time_train_nb.mean(dim=(0,2), keepdim=True)
    time_sd_nb = X_time_train_nb.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train_nb = (X_time_train_nb - time_mu_nb) / time_sd_nb
    X_time_test_nb  = (X_time_test_nb  - time_mu_nb) / time_sd_nb

    bp_mu_nb = X_bp_train_nb.mean(dim=0, keepdim=True)
    bp_sd_nb = X_bp_train_nb.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_nb = (X_bp_train_nb - bp_mu_nb) / bp_sd_nb
    X_bp_test_nb  = (X_bp_test_nb  - bp_mu_nb) / bp_sd_nb

    X_roi_train_nb = torch.zeros((X_bp_train_nb.shape[0], 0))
    X_roi_test_nb  = torch.zeros((X_bp_test_nb.shape[0], 0))

    # =========================
    # HTC
    # =========================
    X_time_train_htc = X_time_t_htc[train_idx_htc].clone()
    X_time_test_htc  = X_time_t_htc[test_idx_htc].clone()

    X_bp_train_htc = X_bp_t_htc[train_idx_htc].clone()
    X_bp_test_htc  = X_bp_t_htc[test_idx_htc].clone()

    X_cov_train_htc = X_cov_t_htc[train_idx_htc].clone()
    X_cov_test_htc  = X_cov_t_htc[test_idx_htc].clone()

    X_plv_train_htc = X_plv_t_htc[train_idx_htc].clone()
    X_plv_test_htc  = X_plv_t_htc[test_idx_htc].clone()

    y_train_htc = y_t_htc[train_idx_htc].clone()
    y_test_htc  = y_t_htc[test_idx_htc].clone()

    time_mu_htc = X_time_train_htc.mean(dim=(0,2), keepdim=True)
    time_sd_htc = X_time_train_htc.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train_htc = (X_time_train_htc - time_mu_htc) / time_sd_htc
    X_time_test_htc  = (X_time_test_htc  - time_mu_htc) / time_sd_htc

    bp_mu_htc = X_bp_train_htc.mean(dim=0, keepdim=True)
    bp_sd_htc = X_bp_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_htc = (X_bp_train_htc - bp_mu_htc) / bp_sd_htc
    X_bp_test_htc  = (X_bp_test_htc  - bp_mu_htc) / bp_sd_htc

    cov_mu_htc = X_cov_train_htc.mean(dim=0, keepdim=True)
    cov_sd_htc = X_cov_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_cov_train_htc = (X_cov_train_htc - cov_mu_htc) / cov_sd_htc
    X_cov_test_htc  = (X_cov_test_htc  - cov_mu_htc) / cov_sd_htc

    X_roi_train_htc = torch.zeros((X_bp_train_htc.shape[0], 0))
    X_roi_test_htc  = torch.zeros((X_bp_test_htc.shape[0], 0))

    # =========================
    # DATASETS
    # =========================
    train_ds_nb = EEGDataset(
        X_time_train_nb, X_bp_train_nb, X_cov_train_nb, X_plv_train_nb, X_roi_train_nb,
        y_train_nb,
        subjects_nb_prefixed[train_idx_nb],
        dataset_id=NBACK_ID
    )

    train_ds_htc = EEGDataset(
        X_time_train_htc, X_bp_train_htc, X_cov_train_htc, X_plv_train_htc, X_roi_train_htc,
        y_train_htc,
        subjects_htc_prefixed[train_idx_htc],
        dataset_id=HTC_ID
    )

    # =========================
    # MERGE
    # =========================
    train_ds_joint = ConcatDataset([train_ds_nb, train_ds_htc])

    train_loader = DataLoader(
        train_ds_joint,
        batch_size=BATCH,
        shuffle=True
    )

    # =========================
    # SANITY CHECK
    # =========================
    batch0 = next(iter(train_loader))
    xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db = batch0

    unique_db, counts_db = torch.unique(db, return_counts=True)
    print("Joint batch dataset distribution:", dict(zip(unique_db.tolist(), counts_db.tolist())))

    break  # only check first fold


JOINT FOLD 1/13
N-back test subject: subject_01
HTC test subject:    subject_01
Joint batch dataset distribution: {0: 124, 1: 272}


In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10
FEW_SHOT_PER_CLASS = 20

N_FOLDS = min(len(all_subjects), len(all_subjects_htc))

overall_results_joint = []

for fold_i in range(N_FOLDS):

    test_subj_nb  = all_subjects[fold_i]
    test_subj_htc = all_subjects_htc[fold_i]

    print("\n" + "="*90)
    print(f"JOINT FOLD {fold_i+1}/{N_FOLDS}")
    print(f"N-back test subject: {test_subj_nb}")
    print(f"HTC test subject:    {test_subj_htc}")
    print("="*90)

    # =========================
    # SPLITS
    # =========================
    test_mask_nb  = (subjects == test_subj_nb)
    train_mask_nb = ~test_mask_nb
    train_idx_nb = np.where(train_mask_nb)[0]
    test_idx_nb  = np.where(test_mask_nb)[0]

    test_mask_htc  = (subjects_htc == test_subj_htc)
    train_mask_htc = ~test_mask_htc
    train_idx_htc = np.where(train_mask_htc)[0]
    test_idx_htc  = np.where(test_mask_htc)[0]

    # =========================
    # N-BACK FEATURE PREP
    # =========================
    X_cov_train_nb = torch.zeros_like(X_cov_t[train_idx_nb])
    X_cov_test_nb  = torch.zeros_like(X_cov_t[test_idx_nb])

    X_plv_train_nb = X_plv_t[train_idx_nb].clone()
    X_plv_test_nb  = X_plv_t[test_idx_nb].clone()

    X_time_train_nb = X_time_t[train_idx_nb].clone()
    X_time_test_nb  = X_time_t[test_idx_nb].clone()

    X_bp_train_nb = X_bp_t[train_idx_nb].clone()
    X_bp_test_nb  = X_bp_t[test_idx_nb].clone()

    y_train_nb = y_t[train_idx_nb].clone()
    y_test_nb  = y_t[test_idx_nb].clone()

    time_mu_nb = X_time_train_nb.mean(dim=(0,2), keepdim=True)
    time_sd_nb = X_time_train_nb.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train_nb = (X_time_train_nb - time_mu_nb) / time_sd_nb
    X_time_test_nb  = (X_time_test_nb  - time_mu_nb) / time_sd_nb

    bp_mu_nb = X_bp_train_nb.mean(dim=0, keepdim=True)
    bp_sd_nb = X_bp_train_nb.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_nb = (X_bp_train_nb - bp_mu_nb) / bp_sd_nb
    X_bp_test_nb  = (X_bp_test_nb  - bp_mu_nb) / bp_sd_nb

    X_roi_train_nb = torch.zeros((X_bp_train_nb.shape[0], 0))
    X_roi_test_nb  = torch.zeros((X_bp_test_nb.shape[0], 0))

    # =========================
    # HTC FEATURE PREP
    # =========================
    X_time_train_htc = X_time_t_htc[train_idx_htc].clone()
    X_time_test_htc  = X_time_t_htc[test_idx_htc].clone()

    X_bp_train_htc = X_bp_t_htc[train_idx_htc].clone()
    X_bp_test_htc  = X_bp_t_htc[test_idx_htc].clone()

    X_cov_train_htc = X_cov_t_htc[train_idx_htc].clone()
    X_cov_test_htc  = X_cov_t_htc[test_idx_htc].clone()

    X_plv_train_htc = X_plv_t_htc[train_idx_htc].clone()
    X_plv_test_htc  = X_plv_t_htc[test_idx_htc].clone()

    y_train_htc = y_t_htc[train_idx_htc].clone()
    y_test_htc  = y_t_htc[test_idx_htc].clone()

    time_mu_htc = X_time_train_htc.mean(dim=(0,2), keepdim=True)
    time_sd_htc = X_time_train_htc.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train_htc = (X_time_train_htc - time_mu_htc) / time_sd_htc
    X_time_test_htc  = (X_time_test_htc  - time_mu_htc) / time_sd_htc

    bp_mu_htc = X_bp_train_htc.mean(dim=0, keepdim=True)
    bp_sd_htc = X_bp_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_htc = (X_bp_train_htc - bp_mu_htc) / bp_sd_htc
    X_bp_test_htc  = (X_bp_test_htc  - bp_mu_htc) / bp_sd_htc

    cov_mu_htc = X_cov_train_htc.mean(dim=0, keepdim=True)
    cov_sd_htc = X_cov_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_cov_train_htc = (X_cov_train_htc - cov_mu_htc) / cov_sd_htc
    X_cov_test_htc  = (X_cov_test_htc  - cov_mu_htc) / cov_sd_htc

    X_roi_train_htc = torch.zeros((X_bp_train_htc.shape[0], 0))
    X_roi_test_htc  = torch.zeros((X_bp_test_htc.shape[0], 0))

    # =========================
    # DATASETS
    # =========================
    train_ds_nb = EEGDataset(
        X_time_train_nb, X_bp_train_nb, X_cov_train_nb, X_plv_train_nb, X_roi_train_nb,
        y_train_nb,
        subjects_nb_prefixed[train_idx_nb],
        dataset_id=NBACK_ID
    )

    test_ds_nb = EEGDataset(
        X_time_test_nb, X_bp_test_nb, X_cov_test_nb, X_plv_test_nb, X_roi_test_nb,
        y_test_nb,
        subjects_nb_prefixed[test_idx_nb],
        dataset_id=NBACK_ID
    )

    train_ds_htc = EEGDataset(
        X_time_train_htc, X_bp_train_htc, X_cov_train_htc, X_plv_train_htc, X_roi_train_htc,
        y_train_htc,
        subjects_htc_prefixed[train_idx_htc],
        dataset_id=HTC_ID
    )

    test_ds_htc = EEGDataset(
        X_time_test_htc, X_bp_test_htc, X_cov_test_htc, X_plv_test_htc, X_roi_test_htc,
        y_test_htc,
        subjects_htc_prefixed[test_idx_htc],
        dataset_id=HTC_ID
    )

    # =========================
    # MERGE
    # =========================
    train_ds_joint = ConcatDataset([train_ds_nb, train_ds_htc])

    train_loader = DataLoader(train_ds_joint, batch_size=BATCH, shuffle=True)
    test_loader_nb = DataLoader(test_ds_nb, batch_size=256, shuffle=False)
    test_loader_htc = DataLoader(test_ds_htc, batch_size=256, shuffle=False)

    # sanity
    batch0 = next(iter(train_loader))
    xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db = batch0
    unique_db, counts_db = torch.unique(db, return_counts=True)
    print("Joint batch dataset distribution:", dict(zip(unique_db.tolist(), counts_db.tolist())))

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_state = None
    best_score = -1.0

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():
            e_train_nb, y_train_nb_full = extract_embeddings(
                model, DataLoader(train_ds_nb, batch_size=256, shuffle=False), device
            )
            e_test_nb, y_test_nb_full = extract_embeddings(model, test_loader_nb, device)

            classes_nb = torch.unique(y_train_nb_full)
            protos_nb = torch.stack([
                e_train_nb[y_train_nb_full == c].mean(dim=0)
                for c in classes_nb
            ])

            logits_nb = F.normalize(e_test_nb, dim=1) @ F.normalize(protos_nb, dim=1).T
            pred_nb = logits_nb.argmax(dim=1)

            label_map_nb = {int(c.item()): i for i, c in enumerate(classes_nb)}
            y_test_nb_local = torch.tensor(
                [label_map_nb[int(v.item())] for v in y_test_nb_full],
                device=pred_nb.device
            )
            acc_nb = (pred_nb == y_test_nb_local).float().mean().item()

            e_train_htc, y_train_htc_full = extract_embeddings(
                model, DataLoader(train_ds_htc, batch_size=256, shuffle=False), device
            )
            e_test_htc, y_test_htc_full = extract_embeddings(model, test_loader_htc, device)

            classes_htc = torch.unique(y_train_htc_full)
            protos_htc = torch.stack([
                e_train_htc[y_train_htc_full == c].mean(dim=0)
                for c in classes_htc
            ])

            logits_htc = F.normalize(e_test_htc, dim=1) @ F.normalize(protos_htc, dim=1).T
            pred_htc = logits_htc.argmax(dim=1)

            label_map_htc = {int(c.item()): i for i, c in enumerate(classes_htc)}
            y_test_htc_local = torch.tensor(
                [label_map_htc[int(v.item())] for v in y_test_htc_full],
                device=pred_htc.device
            )
            acc_htc = (pred_htc == y_test_htc_local).float().mean().item()

        score = 0.5 * acc_nb + 0.5 * acc_htc

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | NB {acc_nb:.4f} | HTC {acc_htc:.4f} | avg {score:.4f}")

    model.load_state_dict(best_state)

    # =========================
    # N-BACK FEW SHOT
    # =========================
    print("\n--- N-BACK FEW SHOT ---")

    test_loader_full = DataLoader(test_ds_nb, batch_size=256, shuffle=False)
    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(idx))
        support_idx.append(idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    e_support = F.normalize(all_e[mask].to(device), dim=1)
    y_support = all_y[mask].to(device)
    e_query   = F.normalize(all_e[~mask].to(device), dim=1)
    y_query   = all_y[~mask].to(device)

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])

    dists = torch.cdist(e_query, protos)
    pred = dists.argmin(dim=1)
    acc_nb_fs = (pred == y_query).float().mean().item()
    print("NB few-shot acc:", acc_nb_fs)

    # =========================
    # HTC FEW SHOT
    # =========================
    print("\n--- HTC FEW SHOT ---")

    test_loader_full = DataLoader(test_ds_htc, batch_size=256, shuffle=False)
    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(idx))
        support_idx.append(idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    e_support = F.normalize(all_e[mask].to(device), dim=1)
    y_support = all_y[mask].to(device)
    e_query   = F.normalize(all_e[~mask].to(device), dim=1)
    y_query   = all_y[~mask].to(device)

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])

    dists = torch.cdist(e_query, protos)
    pred = dists.argmin(dim=1)
    acc_htc_fs = (pred == y_query).float().mean().item()
    print("HTC few-shot acc:", acc_htc_fs)

    overall_results_joint.append({
        "fold": fold_i + 1,
        "test_subj_nb": test_subj_nb,
        "test_subj_htc": test_subj_htc,
        "best_zero_shot_nb": acc_nb,
        "best_zero_shot_htc": acc_htc,
        "fewshot_nb": acc_nb_fs,
        "fewshot_htc": acc_htc_fs
    })


JOINT FOLD 1/13
N-back test subject: subject_01
HTC test subject:    subject_01
Joint batch dataset distribution: {0: 105, 1: 291}
Epoch 01 | NB 0.6488 | HTC 0.6971 | avg 0.6729
Epoch 02 | NB 0.7266 | HTC 0.7166 | avg 0.7216
Epoch 03 | NB 0.7340 | HTC 0.6938 | avg 0.7139
Epoch 04 | NB 0.7014 | HTC 0.7003 | avg 0.7008
Epoch 05 | NB 0.7455 | HTC 0.7459 | avg 0.7457
Epoch 06 | NB 0.7382 | HTC 0.6450 | avg 0.6916
Epoch 07 | NB 0.7035 | HTC 0.6873 | avg 0.6954
Epoch 08 | NB 0.7539 | HTC 0.6352 | avg 0.6946
Epoch 09 | NB 0.7466 | HTC 0.7655 | avg 0.7560
Epoch 10 | NB 0.7361 | HTC 0.5342 | avg 0.6351

--- N-BACK FEW SHOT ---
NB few-shot acc: 0.7878788113594055

--- HTC FEW SHOT ---
HTC few-shot acc: 0.7565543055534363

JOINT FOLD 2/13
N-back test subject: subject_02
HTC test subject:    subject_02
Joint batch dataset distribution: {0: 120, 1: 276}
Epoch 01 | NB 0.4622 | HTC 0.5552 | avg 0.5087
Epoch 02 | NB 0.4473 | HTC 0.3913 | avg 0.4193
Epoch 03 | NB 0.4334 | HTC 0.4783 | avg 0.4559
Epoch

In [ ]:
def supcon_cross_dataset_loss(proj, y, subj, dataset, temperature=0.1):
    """
    proj: (B, D)
    y: (B,)
    subj: (B,)
    dataset: (B,)   # NEW
    """

    proj = F.normalize(proj, dim=1)
    B = proj.shape[0]

    sim = proj @ proj.T  # (B, B)
    sim = sim / temperature

    # masks
    y = y.unsqueeze(0)
    subj = subj.unsqueeze(0)
    dataset = dataset.unsqueeze(0)

    same_label = (y == y.T)
    diff_subj  = (subj != subj.T)
    same_dataset = (dataset == dataset.T)
    cross_dataset = (dataset != dataset.T)

    # ----------------------------
    # POSITIVES
    # ----------------------------

    # within-dataset positives (safe)
    pos_within = same_label & diff_subj & same_dataset

    # cross-dataset positives (CONTROLLED)
    # only align LOW workload (~label 0)
    pos_cross = (y == 0) & (y.T == 0) & diff_subj & cross_dataset

    positives = pos_within | pos_cross

    # ----------------------------
    # NEGATIVES
    # ----------------------------
    negatives = ~same_label

    # remove self
    eye = torch.eye(B, dtype=torch.bool, device=proj.device)
    positives = positives & ~eye
    negatives = negatives & ~eye

    # ----------------------------
    # LOGITS
    # ----------------------------
    exp_sim = torch.exp(sim)

    pos_exp = exp_sim * positives
    neg_exp = exp_sim * negatives

    denom = pos_exp.sum(dim=1) + neg_exp.sum(dim=1) + 1e-8
    log_prob = torch.log((pos_exp.sum(dim=1) + 1e-8) / denom)

    loss = -log_prob.mean()

    return loss

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10
FEW_SHOT_PER_CLASS = 20

N_FOLDS = min(len(all_subjects), len(all_subjects_htc))

overall_results_joint = []

for fold_i in range(N_FOLDS):

    test_subj_nb  = all_subjects[fold_i]
    test_subj_htc = all_subjects_htc[fold_i]

    print("\n" + "="*90)
    print(f"JOINT FOLD {fold_i+1}/{N_FOLDS}")
    print(f"N-back test subject: {test_subj_nb}")
    print(f"HTC test subject:    {test_subj_htc}")
    print("="*90)

    # =========================
    # SPLITS
    # =========================
    test_mask_nb  = (subjects == test_subj_nb)
    train_mask_nb = ~test_mask_nb
    train_idx_nb = np.where(train_mask_nb)[0]
    test_idx_nb  = np.where(test_mask_nb)[0]

    test_mask_htc  = (subjects_htc == test_subj_htc)
    train_mask_htc = ~test_mask_htc
    train_idx_htc = np.where(train_mask_htc)[0]
    test_idx_htc  = np.where(test_mask_htc)[0]

    # =========================
    # N-BACK FEATURE PREP
    # =========================
    X_cov_train_nb = torch.zeros_like(X_cov_t[train_idx_nb])
    X_cov_test_nb  = torch.zeros_like(X_cov_t[test_idx_nb])

    X_plv_train_nb = X_plv_t[train_idx_nb].clone()
    X_plv_test_nb  = X_plv_t[test_idx_nb].clone()

    X_time_train_nb = X_time_t[train_idx_nb].clone()
    X_time_test_nb  = X_time_t[test_idx_nb].clone()

    X_bp_train_nb = X_bp_t[train_idx_nb].clone()
    X_bp_test_nb  = X_bp_t[test_idx_nb].clone()

    y_train_nb = y_t[train_idx_nb].clone()
    y_test_nb  = y_t[test_idx_nb].clone()

    time_mu_nb = X_time_train_nb.mean(dim=(0,2), keepdim=True)
    time_sd_nb = X_time_train_nb.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train_nb = (X_time_train_nb - time_mu_nb) / time_sd_nb
    X_time_test_nb  = (X_time_test_nb  - time_mu_nb) / time_sd_nb

    bp_mu_nb = X_bp_train_nb.mean(dim=0, keepdim=True)
    bp_sd_nb = X_bp_train_nb.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_nb = (X_bp_train_nb - bp_mu_nb) / bp_sd_nb
    X_bp_test_nb  = (X_bp_test_nb  - bp_mu_nb) / bp_sd_nb

    X_roi_train_nb = torch.zeros((X_bp_train_nb.shape[0], 0))
    X_roi_test_nb  = torch.zeros((X_bp_test_nb.shape[0], 0))

    # =========================
    # HTC FEATURE PREP
    # =========================
    X_time_train_htc = X_time_t_htc[train_idx_htc].clone()
    X_time_test_htc  = X_time_t_htc[test_idx_htc].clone()

    X_bp_train_htc = X_bp_t_htc[train_idx_htc].clone()
    X_bp_test_htc  = X_bp_t_htc[test_idx_htc].clone()

    X_cov_train_htc = X_cov_t_htc[train_idx_htc].clone()
    X_cov_test_htc  = X_cov_t_htc[test_idx_htc].clone()

    X_plv_train_htc = X_plv_t_htc[train_idx_htc].clone()
    X_plv_test_htc  = X_plv_t_htc[test_idx_htc].clone()

    y_train_htc = y_t_htc[train_idx_htc].clone()
    y_test_htc  = y_t_htc[test_idx_htc].clone()

    time_mu_htc = X_time_train_htc.mean(dim=(0,2), keepdim=True)
    time_sd_htc = X_time_train_htc.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train_htc = (X_time_train_htc - time_mu_htc) / time_sd_htc
    X_time_test_htc  = (X_time_test_htc  - time_mu_htc) / time_sd_htc

    bp_mu_htc = X_bp_train_htc.mean(dim=0, keepdim=True)
    bp_sd_htc = X_bp_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_htc = (X_bp_train_htc - bp_mu_htc) / bp_sd_htc
    X_bp_test_htc  = (X_bp_test_htc  - bp_mu_htc) / bp_sd_htc

    cov_mu_htc = X_cov_train_htc.mean(dim=0, keepdim=True)
    cov_sd_htc = X_cov_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_cov_train_htc = (X_cov_train_htc - cov_mu_htc) / cov_sd_htc
    X_cov_test_htc  = (X_cov_test_htc  - cov_mu_htc) / cov_sd_htc

    X_roi_train_htc = torch.zeros((X_bp_train_htc.shape[0], 0))
    X_roi_test_htc  = torch.zeros((X_bp_test_htc.shape[0], 0))

    # =========================
    # DATASETS
    # =========================
    train_ds_nb = EEGDataset(
        X_time_train_nb, X_bp_train_nb, X_cov_train_nb, X_plv_train_nb, X_roi_train_nb,
        y_train_nb,
        subjects_nb_prefixed[train_idx_nb],
        dataset_id=NBACK_ID
    )

    test_ds_nb = EEGDataset(
        X_time_test_nb, X_bp_test_nb, X_cov_test_nb, X_plv_test_nb, X_roi_test_nb,
        y_test_nb,
        subjects_nb_prefixed[test_idx_nb],
        dataset_id=NBACK_ID
    )

    train_ds_htc = EEGDataset(
        X_time_train_htc, X_bp_train_htc, X_cov_train_htc, X_plv_train_htc, X_roi_train_htc,
        y_train_htc,
        subjects_htc_prefixed[train_idx_htc],
        dataset_id=HTC_ID
    )

    test_ds_htc = EEGDataset(
        X_time_test_htc, X_bp_test_htc, X_cov_test_htc, X_plv_test_htc, X_roi_test_htc,
        y_test_htc,
        subjects_htc_prefixed[test_idx_htc],
        dataset_id=HTC_ID
    )

    # =========================
    # MERGE
    # =========================
    train_ds_joint = ConcatDataset([train_ds_nb, train_ds_htc])

    train_loader = DataLoader(train_ds_joint, batch_size=BATCH, shuffle=True)
    test_loader_nb = DataLoader(test_ds_nb, batch_size=256, shuffle=False)
    test_loader_htc = DataLoader(test_ds_htc, batch_size=256, shuffle=False)

    # sanity
    batch0 = next(iter(train_loader))
    xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db = batch0
    unique_db, counts_db = torch.unique(db, return_counts=True)
    print("Joint batch dataset distribution:", dict(zip(unique_db.tolist(), counts_db.tolist())))

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_state = None
    best_score = -1.0

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)
            db = db.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            # loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_con = supcon_cross_dataset_loss(proj, yb, sb, db, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():
            e_train_nb, y_train_nb_full = extract_embeddings(
                model, DataLoader(train_ds_nb, batch_size=256, shuffle=False), device
            )
            e_test_nb, y_test_nb_full = extract_embeddings(model, test_loader_nb, device)

            classes_nb = torch.unique(y_train_nb_full)
            protos_nb = torch.stack([
                e_train_nb[y_train_nb_full == c].mean(dim=0)
                for c in classes_nb
            ])

            logits_nb = F.normalize(e_test_nb, dim=1) @ F.normalize(protos_nb, dim=1).T
            pred_nb = logits_nb.argmax(dim=1)

            label_map_nb = {int(c.item()): i for i, c in enumerate(classes_nb)}
            y_test_nb_local = torch.tensor(
                [label_map_nb[int(v.item())] for v in y_test_nb_full],
                device=pred_nb.device
            )
            acc_nb = (pred_nb == y_test_nb_local).float().mean().item()

            e_train_htc, y_train_htc_full = extract_embeddings(
                model, DataLoader(train_ds_htc, batch_size=256, shuffle=False), device
            )
            e_test_htc, y_test_htc_full = extract_embeddings(model, test_loader_htc, device)

            classes_htc = torch.unique(y_train_htc_full)
            protos_htc = torch.stack([
                e_train_htc[y_train_htc_full == c].mean(dim=0)
                for c in classes_htc
            ])

            logits_htc = F.normalize(e_test_htc, dim=1) @ F.normalize(protos_htc, dim=1).T
            pred_htc = logits_htc.argmax(dim=1)

            label_map_htc = {int(c.item()): i for i, c in enumerate(classes_htc)}
            y_test_htc_local = torch.tensor(
                [label_map_htc[int(v.item())] for v in y_test_htc_full],
                device=pred_htc.device
            )
            acc_htc = (pred_htc == y_test_htc_local).float().mean().item()

        score = 0.5 * acc_nb + 0.5 * acc_htc

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | NB {acc_nb:.4f} | HTC {acc_htc:.4f} | avg {score:.4f}")

    model.load_state_dict(best_state)

    # =========================
    # N-BACK FEW SHOT
    # =========================
    print("\n--- N-BACK FEW SHOT ---")

    test_loader_full = DataLoader(test_ds_nb, batch_size=256, shuffle=False)
    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(idx))
        support_idx.append(idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    e_support = F.normalize(all_e[mask].to(device), dim=1)
    y_support = all_y[mask].to(device)
    e_query   = F.normalize(all_e[~mask].to(device), dim=1)
    y_query   = all_y[~mask].to(device)

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])

    dists = torch.cdist(e_query, protos)
    pred = dists.argmin(dim=1)
    acc_nb_fs = (pred == y_query).float().mean().item()
    print("NB few-shot acc:", acc_nb_fs)

    # =========================
    # HTC FEW SHOT
    # =========================
    print("\n--- HTC FEW SHOT ---")

    test_loader_full = DataLoader(test_ds_htc, batch_size=256, shuffle=False)
    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(idx))
        support_idx.append(idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    e_support = F.normalize(all_e[mask].to(device), dim=1)
    y_support = all_y[mask].to(device)
    e_query   = F.normalize(all_e[~mask].to(device), dim=1)
    y_query   = all_y[~mask].to(device)

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])

    dists = torch.cdist(e_query, protos)
    pred = dists.argmin(dim=1)
    acc_htc_fs = (pred == y_query).float().mean().item()
    print("HTC few-shot acc:", acc_htc_fs)

    overall_results_joint.append({
        "fold": fold_i + 1,
        "test_subj_nb": test_subj_nb,
        "test_subj_htc": test_subj_htc,
        "best_zero_shot_nb": acc_nb,
        "best_zero_shot_htc": acc_htc,
        "fewshot_nb": acc_nb_fs,
        "fewshot_htc": acc_htc_fs
    })


JOINT FOLD 1/13
N-back test subject: subject_01
HTC test subject:    subject_01
Joint batch dataset distribution: {0: 136, 1: 260}
Epoch 01 | NB 0.6351 | HTC 0.6156 | avg 0.6254
Epoch 02 | NB 0.6961 | HTC 0.6840 | avg 0.6901
Epoch 03 | NB 0.7676 | HTC 0.6710 | avg 0.7193
Epoch 04 | NB 0.7539 | HTC 0.6059 | avg 0.6799
Epoch 05 | NB 0.7560 | HTC 0.6482 | avg 0.7021
Epoch 06 | NB 0.7571 | HTC 0.6254 | avg 0.6913
Epoch 07 | NB 0.7603 | HTC 0.6189 | avg 0.6896
Epoch 08 | NB 0.7655 | HTC 0.5863 | avg 0.6759
Epoch 09 | NB 0.7298 | HTC 0.6287 | avg 0.6792
Epoch 10 | NB 0.7287 | HTC 0.6287 | avg 0.6787

--- N-BACK FEW SHOT ---
NB few-shot acc: 0.796857476234436

--- HTC FEW SHOT ---
HTC few-shot acc: 0.7752808928489685

JOINT FOLD 2/13
N-back test subject: subject_02
HTC test subject:    subject_02
Joint batch dataset distribution: {0: 99, 1: 297}
Epoch 01 | NB 0.4058 | HTC 0.4114 | avg 0.4086
Epoch 02 | NB 0.4473 | HTC 0.5217 | avg 0.4845
Epoch 03 | NB 0.4579 | HTC 0.3980 | avg 0.4280
Epoch 0

In [ ]:
save_path = "/content/drive/MyDrive/eeg_preprocessed_nback.pt"

torch.save({
    "X_time": X_time_t,
    "X_bp": X_bp_t,
    "X_cov": X_cov_t,
    "X_plv": X_plv_t,
    "y": y_t,
    "subjects": subjects
}, save_path)

print("Saved!")

Saved!


In [ ]:
torch.save({
    "X_time": X_time_t_htc,
    "X_bp": X_bp_t_htc,
    "X_cov": X_cov_t_htc,
    "X_plv": X_plv_t_htc,
    "y": y_t_htc,
    "subjects": subjects_htc
}, "/content/drive/MyDrive/eeg_preprocessed_htc.pt")

In [ ]:
data_htc = torch.load(
    "/content/drive/MyDrive/eeg_preprocessed_htc.pt",
    weights_only=False
)

In [ ]:
X_time_t_htc = data_htc["X_time"]
X_bp_t_htc   = data_htc["X_bp"]
X_cov_t_htc  = data_htc["X_cov"]
X_plv_t_htc  = data_htc["X_plv"]
y_t_htc      = data_htc["y"]
subjects_htc = data_htc["subjects"]

In [ ]:
# =========================
# LOAD N-BACK
# =========================
data_nb = torch.load(
    "/content/drive/MyDrive/eeg_preprocessed_nback.pt",
    weights_only=False
)

X_time_t = data_nb["X_time"]
X_bp_t   = data_nb["X_bp"]
X_cov_t  = data_nb["X_cov"]
X_plv_t  = data_nb["X_plv"]
y_t      = data_nb["y"]
subjects = data_nb["subjects"]

print("Loaded N-back")


# =========================
# LOAD HTC
# =========================
data_htc = torch.load(
    "/content/drive/MyDrive/eeg_preprocessed_htc.pt",
    weights_only=False
)

X_time_t_htc = data_htc["X_time"]
X_bp_t_htc   = data_htc["X_bp"]
X_cov_t_htc  = data_htc["X_cov"]
X_plv_t_htc  = data_htc["X_plv"]
y_t_htc      = data_htc["y"]
subjects_htc = data_htc["subjects"]

print("Loaded HTC")

Loaded N-back
Loaded HTC


In [ ]:
print("\n--- N-BACK ---")
print(X_time_t.shape, X_bp_t.shape, X_cov_t.shape, X_plv_t.shape, y_t.shape)

print("\n--- HTC ---")
print(X_time_t_htc.shape, X_bp_t_htc.shape, X_cov_t_htc.shape, X_plv_t_htc.shape, y_t_htc.shape)


--- N-BACK ---
torch.Size([12159, 14, 512]) torch.Size([12159, 42]) torch.Size([15007, 105]) torch.Size([15007, 210]) torch.Size([12159])

--- HTC ---
torch.Size([5056, 14, 512]) torch.Size([5056, 42]) torch.Size([5056, 105]) torch.Size([5056, 210]) torch.Size([5056])


In [ ]:
def extract_embeddings(model, loader, device):
    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            db      = db.to(device)

            # SAFE: supports both versions of model
            try:
                e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi, db)
            except TypeError:
                e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)

In [ ]:
import numpy as np

# ----------------------------
# REMOVE 4 LOWEST HTC SUBJECTS
# ----------------------------
htc_perf = {
    "subject_01": 0.7677902579307556,
    "subject_02": 0.8880308866500854,
    "subject_03": 0.7392995953559875,
    "subject_06": 0.7952755689620972,
    "subject_08": 0.6611570119857788,
    "subject_12": 0.8793774247169495,
    "subject_16": 0.7346153855323792,
    "subject_17": 0.73046875,
    "subject_18": 0.8046875,
    "subject_19": 0.957198441028595,
    "subject_20": 0.6640926599502563,
    "subject_21": 0.8560311198234558,
    "subject_22": 0.8615384697914124,
    "subject_23": 0.8759689927101135,
    "subject_24": 0.9372549653053284,
    "subject_25": 0.6100385785102844,
    "subject_26": 0.8859315514564514,
}

K = 4
sorted_subjs = sorted(htc_perf.items(), key=lambda x: x[1])
remove_subjs = [s for s, _ in sorted_subjs[:K]]

print("Removing HTC subjects:", remove_subjs)

keep_mask = np.array([s not in remove_subjs for s in subjects_htc])

X_time_t_htc = X_time_t_htc[keep_mask]
X_bp_t_htc   = X_bp_t_htc[keep_mask]
X_cov_t_htc  = X_cov_t_htc[keep_mask]
X_plv_t_htc  = X_plv_t_htc[keep_mask]
y_t_htc      = y_t_htc[keep_mask]
subjects_htc = subjects_htc[keep_mask]

all_subjects_htc = np.unique(subjects_htc)
print("Remaining HTC subjects:", len(all_subjects_htc), all_subjects_htc)

Removing HTC subjects: ['subject_25', 'subject_08', 'subject_20', 'subject_17']
Remaining HTC subjects: 13 ['subject_01' 'subject_02' 'subject_03' 'subject_06' 'subject_12'
 'subject_16' 'subject_18' 'subject_19' 'subject_21' 'subject_22'
 'subject_23' 'subject_24' 'subject_26']


In [ ]:
HTC_ID = 0
NBACK_ID = 1

subjects_nb_prefixed  = np.array([f"nb_{s}" for s in subjects])
subjects_htc_prefixed = np.array([f"htc_{s}" for s in subjects_htc])

print("Example N-back subjects:", subjects_nb_prefixed[:5])
print("Example HTC subjects:", subjects_htc_prefixed[:5])

Example N-back subjects: ['nb_subject_01' 'nb_subject_01' 'nb_subject_01' 'nb_subject_01'
 'nb_subject_01']
Example HTC subjects: ['htc_subject_01' 'htc_subject_01' 'htc_subject_01' 'htc_subject_01'
 'htc_subject_01']


In [ ]:
import torch
from torch.utils.data import Dataset

class EEGDataset(Dataset):
    def __init__(self, X_time, X_bp, X_cov, X_plv, X_roi, y, subj_strs, dataset_id):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_cov  = X_cov
        self.X_plv  = X_plv
        self.X_roi  = X_roi
        self.y      = y

        self.dataset_id = int(dataset_id)

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_cov[idx],
            self.X_plv[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx],
            torch.tensor(self.dataset_id, dtype=torch.long)
        )

In [ ]:
import torch
from torch.utils.data import Dataset

class EEGDataset(Dataset):
    def __init__(self, X_time, X_bp, X_cov, X_plv, X_roi, y, subj_strs, dataset_id):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_cov  = X_cov
        self.X_plv  = X_plv
        self.X_roi  = X_roi
        self.y      = y

        self.dataset_id = int(dataset_id)

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_cov[idx],
            self.X_plv[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx],
            torch.tensor(self.dataset_id, dtype=torch.long)
        )

In [ ]:
import torch.nn.functional as F

def supcon_within_dataset_loss(proj, y, subj, dataset, temperature=0.1):
    proj = F.normalize(proj, dim=1)
    B = proj.shape[0]

    sim = (proj @ proj.T) / temperature
    exp_sim = torch.exp(sim)

    y = y.unsqueeze(0)
    subj = subj.unsqueeze(0)
    dataset = dataset.unsqueeze(0)

    same_label   = (y == y.T)
    diff_subj    = (subj != subj.T)
    same_dataset = (dataset == dataset.T)

    positives = same_label & diff_subj & same_dataset

    eye = torch.eye(B, dtype=torch.bool, device=proj.device)
    positives = positives & ~eye
    valid = ~eye

    pos_exp = exp_sim * positives
    all_exp = exp_sim * valid

    pos_sum = pos_exp.sum(dim=1)
    denom   = all_exp.sum(dim=1) + 1e-8

    has_pos = positives.sum(dim=1) > 0
    if has_pos.sum() == 0:
        return torch.tensor(0.0, device=proj.device)

    log_prob = torch.log((pos_sum[has_pos] + 1e-8) / denom[has_pos])
    loss = -log_prob.mean()
    return loss

In [ ]:
from torch.utils.data import Sampler
import random
from collections import defaultdict

class JointBalancedBatchSampler(Sampler):
    def __init__(self, dataset, batch_size, subjects_per_dataset=6):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_dataset = subjects_per_dataset

        self.group_to_indices = defaultdict(list)

        for idx in range(len(dataset)):
            _, _, _, _, _, _, subj, db = dataset[idx]
            key = (int(db.item()), int(subj.item()))
            self.group_to_indices[key].append(idx)

        self.nb_groups  = [g for g in self.group_to_indices if g[0] == NBACK_ID]
        self.htc_groups = [g for g in self.group_to_indices if g[0] == HTC_ID]

        assert len(self.nb_groups) > 0, "No N-back groups found"
        assert len(self.htc_groups) > 0, "No HTC groups found"

        self.nb_total  = sum(len(self.group_to_indices[g]) for g in self.nb_groups)
        self.htc_total = sum(len(self.group_to_indices[g]) for g in self.htc_groups)

        self.max_samples = max(self.nb_total, self.htc_total)
        self.num_batches = self.max_samples // batch_size

        self.samples_per_subject = batch_size // (2 * subjects_per_dataset)

    def __iter__(self):
        group_lists = {}
        for g in self.group_to_indices:
            lst = self.group_to_indices[g][:]
            random.shuffle(lst)
            group_lists[g] = lst

        group_ptrs = {g: 0 for g in group_lists}

        for _ in range(self.num_batches):
            chosen_nb  = random.sample(self.nb_groups,  self.subjects_per_dataset)
            chosen_htc = random.sample(self.htc_groups, self.subjects_per_dataset)

            batch = []

            for g in chosen_nb + chosen_htc:
                for _ in range(self.samples_per_subject):
                    if group_ptrs[g] >= len(group_lists[g]):
                        random.shuffle(group_lists[g])
                        group_ptrs[g] = 0

                    batch.append(group_lists[g][group_ptrs[g]])
                    group_ptrs[g] += 1

            yield batch

    def __len__(self):
        return self.num_batches

In [ ]:
def mixed_dataset_metric_loss(e, proj, yb, sb, db,
                              lambda_con=0.5,
                              proto_lambda=0.10,
                              temperature=0.1):
    loss_metric = torch.tensor(0.0, device=e.device)
    loss_proto  = torch.tensor(0.0, device=e.device)

    mask_nb  = (db == NBACK_ID)
    mask_htc = (db == HTC_ID)

    if mask_nb.any():
        loss_metric = loss_metric + prototype_ce_loss(
            e[mask_nb], yb[mask_nb], temperature=temperature
        )
        loss_proto = loss_proto + prototype_loss(
            e[mask_nb], yb[mask_nb]
        )

    if mask_htc.any():
        loss_metric = loss_metric + prototype_ce_loss(
            e[mask_htc], yb[mask_htc], temperature=temperature
        )
        loss_proto = loss_proto + prototype_loss(
            e[mask_htc], yb[mask_htc]
        )

    loss_con = supcon_within_dataset_loss(
        proj, yb, sb, db, temperature=temperature
    )

    total = loss_metric + lambda_con * loss_con + proto_lambda * loss_proto
    return total, loss_metric, loss_con, loss_proto

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4
SHIFT_THRESH = 0.15

N_FOLDS = min(len(all_subjects), len(all_subjects_htc))
overall_results_joint = []

for fold_i in range(N_FOLDS):

    test_subj_nb  = all_subjects[fold_i]
    test_subj_htc = all_subjects_htc[fold_i]

    print("\n" + "="*90)
    print(f"JOINT FOLD {fold_i+1}/{N_FOLDS}")
    print(f"N-back test subject: {test_subj_nb}")
    print(f"HTC test subject:    {test_subj_htc}")
    print("="*90)

    # =========================
    # SPLIT
    # =========================
    test_mask_nb  = (subjects == test_subj_nb)
    train_mask_nb = ~test_mask_nb
    train_idx_nb  = np.where(train_mask_nb)[0]
    test_idx_nb   = np.where(test_mask_nb)[0]

    test_mask_htc  = (subjects_htc == test_subj_htc)
    train_mask_htc = ~test_mask_htc
    train_idx_htc  = np.where(train_mask_htc)[0]
    test_idx_htc   = np.where(test_mask_htc)[0]

    # =========================
    # N-BACK FEATURE PREP
    # =========================
    X_cov_train_nb = torch.zeros_like(X_cov_t[train_idx_nb])
    X_cov_test_nb  = torch.zeros_like(X_cov_t[test_idx_nb])

    X_plv_train_nb = X_plv_t[train_idx_nb].clone()
    X_plv_test_nb  = X_plv_t[test_idx_nb].clone()

    X_time_train_nb = X_time_t[train_idx_nb].clone()
    X_time_test_nb  = X_time_t[test_idx_nb].clone()

    X_bp_train_nb = X_bp_t[train_idx_nb].clone()
    X_bp_test_nb  = X_bp_t[test_idx_nb].clone()

    y_train_nb = y_t[train_idx_nb].clone()
    y_test_nb  = y_t[test_idx_nb].clone()

    time_mu_nb = X_time_train_nb.mean(dim=(0, 2), keepdim=True)
    time_sd_nb = X_time_train_nb.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_nb = (X_time_train_nb - time_mu_nb) / time_sd_nb
    X_time_test_nb  = (X_time_test_nb  - time_mu_nb) / time_sd_nb

    bp_mu_nb = X_bp_train_nb.mean(dim=0, keepdim=True)
    bp_sd_nb = X_bp_train_nb.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_nb = (X_bp_train_nb - bp_mu_nb) / bp_sd_nb
    X_bp_test_nb  = (X_bp_test_nb  - bp_mu_nb) / bp_sd_nb

    X_roi_train_nb = torch.zeros((X_bp_train_nb.shape[0], 0))
    X_roi_test_nb  = torch.zeros((X_bp_test_nb.shape[0], 0))

    # =========================
    # HTC FEATURE PREP
    # =========================
    X_time_train_htc = X_time_t_htc[train_idx_htc].clone()
    X_time_test_htc  = X_time_t_htc[test_idx_htc].clone()

    X_bp_train_htc = X_bp_t_htc[train_idx_htc].clone()
    X_bp_test_htc  = X_bp_t_htc[test_idx_htc].clone()

    X_cov_train_htc = X_cov_t_htc[train_idx_htc].clone()
    X_cov_test_htc  = X_cov_t_htc[test_idx_htc].clone()

    X_plv_train_htc = X_plv_t_htc[train_idx_htc].clone()
    X_plv_test_htc  = X_plv_t_htc[test_idx_htc].clone()

    y_train_htc = y_t_htc[train_idx_htc].clone()
    y_test_htc  = y_t_htc[test_idx_htc].clone()

    time_mu_htc = X_time_train_htc.mean(dim=(0, 2), keepdim=True)
    time_sd_htc = X_time_train_htc.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_htc = (X_time_train_htc - time_mu_htc) / time_sd_htc
    X_time_test_htc  = (X_time_test_htc  - time_mu_htc) / time_sd_htc

    bp_mu_htc = X_bp_train_htc.mean(dim=0, keepdim=True)
    bp_sd_htc = X_bp_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_htc = (X_bp_train_htc - bp_mu_htc) / bp_sd_htc
    X_bp_test_htc  = (X_bp_test_htc  - bp_mu_htc) / bp_sd_htc

    cov_mu_htc = X_cov_train_htc.mean(dim=0, keepdim=True)
    cov_sd_htc = X_cov_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_cov_train_htc = (X_cov_train_htc - cov_mu_htc) / cov_sd_htc
    X_cov_test_htc  = (X_cov_test_htc  - cov_mu_htc) / cov_sd_htc

    X_roi_train_htc = torch.zeros((X_bp_train_htc.shape[0], 0))
    X_roi_test_htc  = torch.zeros((X_bp_test_htc.shape[0], 0))

    # =========================
    # DATASETS
    # =========================
    train_ds_nb = EEGDataset(
        X_time_train_nb, X_bp_train_nb, X_cov_train_nb, X_plv_train_nb, X_roi_train_nb,
        y_train_nb,
        subjects_nb_prefixed[train_idx_nb],
        dataset_id=NBACK_ID
    )

    test_ds_nb = EEGDataset(
        X_time_test_nb, X_bp_test_nb, X_cov_test_nb, X_plv_test_nb, X_roi_test_nb,
        y_test_nb,
        subjects_nb_prefixed[test_idx_nb],
        dataset_id=NBACK_ID
    )

    train_ds_htc = EEGDataset(
        X_time_train_htc, X_bp_train_htc, X_cov_train_htc, X_plv_train_htc, X_roi_train_htc,
        y_train_htc,
        subjects_htc_prefixed[train_idx_htc],
        dataset_id=HTC_ID
    )

    test_ds_htc = EEGDataset(
        X_time_test_htc, X_bp_test_htc, X_cov_test_htc, X_plv_test_htc, X_roi_test_htc,
        y_test_htc,
        subjects_htc_prefixed[test_idx_htc],
        dataset_id=HTC_ID
    )

    train_ds_joint = ConcatDataset([train_ds_nb, train_ds_htc])

    sampler = JointBalancedBatchSampler(
        train_ds_joint,
        batch_size=BATCH,
        subjects_per_dataset=6
    )

    train_loader         = DataLoader(train_ds_joint, batch_sampler=sampler)
    train_eval_loader_nb = DataLoader(train_ds_nb, batch_size=256, shuffle=False)
    train_eval_loader_htc = DataLoader(train_ds_htc, batch_size=256, shuffle=False)
    test_loader_nb       = DataLoader(test_ds_nb, batch_size=256, shuffle=False)
    test_loader_htc      = DataLoader(test_ds_htc, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_score = -1.0
    best_state = None
    best_acc_nb = -1.0
    best_acc_htc = -1.0

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)
            db      = db.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss, loss_metric, loss_con, loss_proto = mixed_dataset_metric_loss(
                e, proj, yb, sb, db,
                lambda_con=LAMBDA,
                proto_lambda=PROTO_LAMBDA,
                temperature=TEMP
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL: NB
        # =========================
        model.eval()
        with torch.no_grad():
            e_train_nb, y_train_nb_full = extract_embeddings(model, train_eval_loader_nb, device)
            e_test_nb,  y_test_nb_full  = extract_embeddings(model, test_loader_nb, device)

            classes_nb = torch.unique(y_train_nb_full)
            protos_nb = torch.stack([
                e_train_nb[y_train_nb_full == c].mean(dim=0) for c in classes_nb
            ])

            logits_nb = F.normalize(e_test_nb, dim=1) @ F.normalize(protos_nb, dim=1).T
            pred_nb = logits_nb.argmax(dim=1)

            label_map_nb = {int(c.item()): i for i, c in enumerate(classes_nb)}
            y_test_nb_local = torch.tensor(
                [label_map_nb[int(v.item())] for v in y_test_nb_full],
                device=pred_nb.device
            )

            acc_nb = (pred_nb == y_test_nb_local).float().mean().item()

            # =========================
            # ZERO-SHOT EVAL: HTC
            # =========================
            e_train_htc, y_train_htc_full = extract_embeddings(model, train_eval_loader_htc, device)
            e_test_htc,  y_test_htc_full  = extract_embeddings(model, test_loader_htc, device)

            classes_htc = torch.unique(y_train_htc_full)
            protos_htc = torch.stack([
                e_train_htc[y_train_htc_full == c].mean(dim=0) for c in classes_htc
            ])

            logits_htc = F.normalize(e_test_htc, dim=1) @ F.normalize(protos_htc, dim=1).T
            pred_htc = logits_htc.argmax(dim=1)

            label_map_htc = {int(c.item()): i for i, c in enumerate(classes_htc)}
            y_test_htc_local = torch.tensor(
                [label_map_htc[int(v.item())] for v in y_test_htc_full],
                device=pred_htc.device
            )

            acc_htc = (pred_htc == y_test_htc_local).float().mean().item()

        score = 0.5 * acc_nb + 0.5 * acc_htc

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            best_acc_nb = acc_nb
            best_acc_htc = acc_htc

        print(f"Epoch {ep:02d} | NB zero-shot {acc_nb:.4f} | HTC zero-shot {acc_htc:.4f} | avg {score:.4f}")

    # =====================================================
    # LOAD BEST BASE MODEL
    # =====================================================
    model.load_state_dict(best_state)
    best_joint_state = copy.deepcopy(best_state)

    # =====================================================
    # SHIFT COMPUTATION ON BEST BASE MODEL
    # =====================================================
    model.eval()
    with torch.no_grad():
        e_train_nb_base, y_train_nb_base = extract_embeddings(model, train_eval_loader_nb, device)
        e_test_nb_base,  y_test_nb_base  = extract_embeddings(model, test_loader_nb, device)

        e_train_htc_base, y_train_htc_base = extract_embeddings(model, train_eval_loader_htc, device)
        e_test_htc_base,  y_test_htc_base  = extract_embeddings(model, test_loader_htc, device)

    shift_nb = torch.norm(e_train_nb_base.mean(0) - e_test_nb_base.mean(0)).item()
    shift_htc = torch.norm(e_train_htc_base.mean(0) - e_test_htc_base.mean(0)).item()

    print(f"Best-model shift | NB: {shift_nb:.4f} | HTC: {shift_htc:.4f}")

    # =====================================================
    # FEW-SHOT: N-BACK (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    # start NB adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_nb = e_test_nb_base.clone()
    all_y_nb = y_test_nb_base.clone()

    support_idx_nb = []
    for c in torch.unique(all_y_nb):
        class_idx = (all_y_nb == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_nb.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_nb = torch.cat(support_idx_nb)

    mask_nb = torch.zeros(len(all_y_nb), dtype=torch.bool)
    mask_nb[support_idx_nb] = True

    support_subset_nb = torch.utils.data.Subset(test_ds_nb, support_idx_nb.tolist())
    support_loader_nb = DataLoader(support_subset_nb, batch_size=256, shuffle=False)

    use_bn_nb = shift_nb > SHIFT_THRESH
    print(f"N-back BN USED: {use_bn_nb}")

    if use_bn_nb:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_nb:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_nb, all_y_nb = extract_embeddings(model, test_loader_nb, device)

    e_support_nb = all_e_nb[mask_nb].to(device)
    y_support_nb = all_y_nb[mask_nb].to(device)
    e_query_nb   = all_e_nb[~mask_nb].to(device)
    y_query_nb   = all_y_nb[~mask_nb].to(device)

    e_support_nb = F.normalize(e_support_nb, dim=1)
    e_query_nb   = F.normalize(e_query_nb, dim=1)

    var_nb = e_support_nb.var(dim=0)
    weights_nb = 1.0 / (var_nb + 1e-6)
    weights_nb = weights_nb / weights_nb.mean()
    e_support_nb = e_support_nb * weights_nb
    e_query_nb   = e_query_nb * weights_nb

    classes_nb = torch.unique(y_support_nb)
    protos_nb = torch.stack([
        e_support_nb[y_support_nb == c].mean(dim=0) for c in classes_nb
    ])
    protos_nb = F.normalize(protos_nb, dim=1)

    residuals_nb = torch.cat([
        e_support_nb[y_support_nb == c] - protos_nb[i]
        for i, c in enumerate(classes_nb)
    ], dim=0)

    D_nb = residuals_nb.shape[1]
    cov_nb = (residuals_nb.t() @ residuals_nb) / max(residuals_nb.shape[0] - 1, 1)

    avg_var_nb = torch.trace(cov_nb) / D_nb
    cov_shrink_nb = (1 - SHRINK) * cov_nb + SHRINK * (avg_var_nb * torch.eye(D_nb, device=device))
    cov_inv_nb = torch.linalg.pinv(cov_shrink_nb)

    diffs_nb = e_query_nb.unsqueeze(1) - protos_nb.unsqueeze(0)
    dists_nb = torch.einsum("nkd,dd,nkd->nk", diffs_nb, cov_inv_nb, diffs_nb)

    pred_nb_fs = dists_nb.argmin(dim=1)
    fewshot_acc_nb = (pred_nb_fs == y_query_nb).float().mean().item()

    print("N-back gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_nb)

    # =====================================================
    # FEW-SHOT: HTC (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(2234 + fold_i)
    np.random.seed(2234 + fold_i)

    # start HTC adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_htc = e_test_htc_base.clone()
    all_y_htc = y_test_htc_base.clone()

    support_idx_htc = []
    for c in torch.unique(all_y_htc):
        class_idx = (all_y_htc == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_htc.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_htc = torch.cat(support_idx_htc)

    mask_htc = torch.zeros(len(all_y_htc), dtype=torch.bool)
    mask_htc[support_idx_htc] = True

    support_subset_htc = torch.utils.data.Subset(test_ds_htc, support_idx_htc.tolist())
    support_loader_htc = DataLoader(support_subset_htc, batch_size=256, shuffle=False)

    use_bn_htc = shift_htc > SHIFT_THRESH
    print(f"HTC BN USED: {use_bn_htc}")

    if use_bn_htc:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_htc:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_htc, all_y_htc = extract_embeddings(model, test_loader_htc, device)

    e_support_htc = all_e_htc[mask_htc].to(device)
    y_support_htc = all_y_htc[mask_htc].to(device)
    e_query_htc   = all_e_htc[~mask_htc].to(device)
    y_query_htc   = all_y_htc[~mask_htc].to(device)

    e_support_htc = F.normalize(e_support_htc, dim=1)
    e_query_htc   = F.normalize(e_query_htc, dim=1)

    var_htc = e_support_htc.var(dim=0)
    weights_htc = 1.0 / (var_htc + 1e-6)
    weights_htc = weights_htc / weights_htc.mean()
    e_support_htc = e_support_htc * weights_htc
    e_query_htc   = e_query_htc * weights_htc

    classes_htc = torch.unique(y_support_htc)
    protos_htc = torch.stack([
        e_support_htc[y_support_htc == c].mean(dim=0) for c in classes_htc
    ])
    protos_htc = F.normalize(protos_htc, dim=1)

    residuals_htc = torch.cat([
        e_support_htc[y_support_htc == c] - protos_htc[i]
        for i, c in enumerate(classes_htc)
    ], dim=0)

    D_htc = residuals_htc.shape[1]
    cov_htc = (residuals_htc.t() @ residuals_htc) / max(residuals_htc.shape[0] - 1, 1)

    avg_var_htc = torch.trace(cov_htc) / D_htc
    cov_shrink_htc = (1 - SHRINK) * cov_htc + SHRINK * (avg_var_htc * torch.eye(D_htc, device=device))
    cov_inv_htc = torch.linalg.pinv(cov_shrink_htc)

    diffs_htc = e_query_htc.unsqueeze(1) - protos_htc.unsqueeze(0)
    dists_htc = torch.einsum("nkd,dd,nkd->nk", diffs_htc, cov_inv_htc, diffs_htc)

    pred_htc_fs = dists_htc.argmin(dim=1)
    fewshot_acc_htc = (pred_htc_fs == y_query_htc).float().mean().item()

    print("HTC gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_htc)

    overall_results_joint.append({
        "fold": fold_i + 1,
        "test_subj_nb": test_subj_nb,
        "test_subj_htc": test_subj_htc,
        "best_zero_shot_nb": best_acc_nb,
        "best_zero_shot_htc": best_acc_htc,
        "shift_nb": shift_nb,
        "shift_htc": shift_htc,
        "bn_used_nb": bool(use_bn_nb),
        "bn_used_htc": bool(use_bn_htc),
        "fewshot_nb": fewshot_acc_nb,
        "fewshot_htc": fewshot_acc_htc
    })

print("\nJOINT METRIC TRAIN + GATED ADABN + MAHALANOBIS LOSO SUMMARY")
for r in overall_results_joint:
    print(
        f"Fold {r['fold']:02d} | "
        f"NB test {r['test_subj_nb']} | HTC test {r['test_subj_htc']} | "
        f"NB zero-shot {r['best_zero_shot_nb']:.4f} | "
        f"HTC zero-shot {r['best_zero_shot_htc']:.4f} | "
        f"NB shift {r['shift_nb']:.4f} | HTC shift {r['shift_htc']:.4f} | "
        f"NB BN {r['bn_used_nb']} | HTC BN {r['bn_used_htc']} | "
        f"NB few-shot {r['fewshot_nb']:.4f} | "
        f"HTC few-shot {r['fewshot_htc']:.4f}"
    )

nb_fs_accs  = [r["fewshot_nb"] for r in overall_results_joint]
htc_fs_accs = [r["fewshot_htc"] for r in overall_results_joint]

print("\nNB few-shot mean/std:", float(np.mean(nb_fs_accs)), float(np.std(nb_fs_accs)))
print("HTC few-shot mean/std:", float(np.mean(htc_fs_accs)), float(np.std(htc_fs_accs)))


JOINT FOLD 1/13
N-back test subject: subject_01
HTC test subject:    subject_01
Epoch 01 | NB zero-shot 0.5584 | HTC zero-shot 0.6189 | avg 0.5886
Epoch 02 | NB zero-shot 0.6078 | HTC zero-shot 0.6808 | avg 0.6443
Epoch 03 | NB zero-shot 0.5899 | HTC zero-shot 0.5798 | avg 0.5849
Epoch 04 | NB zero-shot 0.6551 | HTC zero-shot 0.5831 | avg 0.6191
Epoch 05 | NB zero-shot 0.6320 | HTC zero-shot 0.5798 | avg 0.6059
Epoch 06 | NB zero-shot 0.6583 | HTC zero-shot 0.6450 | avg 0.6516
Epoch 07 | NB zero-shot 0.7077 | HTC zero-shot 0.5212 | avg 0.6144
Epoch 08 | NB zero-shot 0.6772 | HTC zero-shot 0.4691 | avg 0.5731
Epoch 09 | NB zero-shot 0.6919 | HTC zero-shot 0.6417 | avg 0.6668
Epoch 10 | NB zero-shot 0.7098 | HTC zero-shot 0.4788 | avg 0.5943
Best-model shift | NB: 0.1422 | HTC: 0.2301
N-back BN USED: False
N-back gated AdaBN + Mahalanobis few-shot acc: 0.8047138452529907
HTC BN USED: True
HTC gated AdaBN + Mahalanobis few-shot acc: 0.7116104960441589

JOINT FOLD 2/13
N-back test subject

In [ ]:
# =========================================
# JOINT PRETRAINING (NB + HTC)
# =========================================

import torch
from torch.utils.data import DataLoader, ConcatDataset

EPOCHS_PRETRAIN = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

HTC_ID = 0
NBACK_ID = 1

pretrain_states = []

for fold_i in range(min(len(all_subjects), len(all_subjects_htc))):

    test_subj_nb  = all_subjects[fold_i]
    test_subj_htc = all_subjects_htc[fold_i]

    print("\n" + "="*80)
    print(f"PRETRAIN FOLD {fold_i+1}")
    print("="*80)

    # =========================
    # SPLIT (same as before)
    # =========================
    test_mask_nb  = (subjects == test_subj_nb)
    train_mask_nb = ~test_mask_nb
    train_idx_nb = np.where(train_mask_nb)[0]

    test_mask_htc  = (subjects_htc == test_subj_htc)
    train_mask_htc = ~test_mask_htc
    train_idx_htc = np.where(train_mask_htc)[0]

    # =========================
    # NB FEATURES (same as before)
    # =========================
    X_time_nb = X_time_t[train_idx_nb].clone()
    X_bp_nb   = X_bp_t[train_idx_nb].clone()
    X_cov_nb  = torch.zeros_like(X_cov_t[train_idx_nb])
    X_plv_nb  = X_plv_t[train_idx_nb].clone()
    y_nb      = y_t[train_idx_nb].clone()

    time_mu = X_time_nb.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_nb.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_nb = (X_time_nb - time_mu) / time_sd

    bp_mu = X_bp_nb.mean(dim=0, keepdim=True)
    bp_sd = X_bp_nb.std(dim=0, keepdim=True) + 1e-6
    X_bp_nb = (X_bp_nb - bp_mu) / bp_sd

    X_roi_nb = torch.zeros((X_bp_nb.shape[0], 0))

    # =========================
    # HTC FEATURES
    # =========================
    X_time_htc = X_time_t_htc[train_idx_htc].clone()
    X_bp_htc   = X_bp_t_htc[train_idx_htc].clone()
    X_cov_htc  = X_cov_t_htc[train_idx_htc].clone()
    X_plv_htc  = X_plv_t_htc[train_idx_htc].clone()
    y_htc      = y_t_htc[train_idx_htc].clone()

    time_mu = X_time_htc.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_htc.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_htc = (X_time_htc - time_mu) / time_sd

    bp_mu = X_bp_htc.mean(dim=0, keepdim=True)
    bp_sd = X_bp_htc.std(dim=0, keepdim=True) + 1e-6
    X_bp_htc = (X_bp_htc - bp_mu) / bp_sd

    cov_mu = X_cov_htc.mean(dim=0, keepdim=True)
    cov_sd = X_cov_htc.std(dim=0, keepdim=True) + 1e-6
    X_cov_htc = (X_cov_htc - cov_mu) / cov_sd

    X_roi_htc = torch.zeros((X_bp_htc.shape[0], 0))

    # =========================
    # DATASETS
    # =========================
    train_ds_nb = EEGDataset(
        X_time_nb, X_bp_nb, X_cov_nb, X_plv_nb, X_roi_nb,
        y_nb, subjects_nb_prefixed[train_idx_nb], NBACK_ID
    )

    train_ds_htc = EEGDataset(
        X_time_htc, X_bp_htc, X_cov_htc, X_plv_htc, X_roi_htc,
        y_htc, subjects_htc_prefixed[train_idx_htc], HTC_ID
    )

    train_ds = ConcatDataset([train_ds_nb, train_ds_htc])

    loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    # =========================
    # TRAIN
    # =========================
    for ep in range(EPOCHS_PRETRAIN):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)
            db      = db.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss, *_ = mixed_dataset_metric_loss(
                e, proj, yb, sb, db,
                lambda_con=LAMBDA,
                proto_lambda=PROTO_LAMBDA,
                temperature=TEMP
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        print(f"Epoch {ep+1} done")

    # SAVE PER-FOLD
    torch.save(model.state_dict(), f"pretrained_fold_{fold_i}.pt")


PRETRAIN FOLD 1
Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
Epoch 6 done
Epoch 7 done
Epoch 8 done
Epoch 9 done
Epoch 10 done

PRETRAIN FOLD 2
Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
Epoch 6 done
Epoch 7 done
Epoch 8 done
Epoch 9 done
Epoch 10 done

PRETRAIN FOLD 3
Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
Epoch 6 done
Epoch 7 done
Epoch 8 done
Epoch 9 done
Epoch 10 done

PRETRAIN FOLD 4
Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
Epoch 6 done
Epoch 7 done
Epoch 8 done
Epoch 9 done
Epoch 10 done

PRETRAIN FOLD 5
Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
Epoch 6 done
Epoch 7 done
Epoch 8 done
Epoch 9 done
Epoch 10 done

PRETRAIN FOLD 6
Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
Epoch 6 done
Epoch 7 done
Epoch 8 done
Epoch 9 done
Epoch 10 done

PRETRAIN FOLD 7
Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
Epoch 6 done
Epoch 7 done
Epoc

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

NBACK_ID = 1  # ✅ ADD THIS

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train,
        y_train, subj_train,
        dataset_id=NBACK_ID   # ✅ FIX HERE
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test, X_plv_test, X_roi_test,
        y_test, subj_test,
        dataset_id=NBACK_ID   # ✅ FIX HERE
    )

    sampler = SubjectBalancedBatchSampler(
        train_ds,
        batch_size=BATCH,
        subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    # ✅ LOAD PRETRAINED WEIGHTS
    model.load_state_dict(torch.load(f"pretrained_fold_{fold_i}.pt"))

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            # ⚠️ db exists but we IGNORE it here

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=TEMP)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)

    # =========================
    # FEW-SHOT (UNCHANGED)
    # =========================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)
    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat(
        [e_support[y_support == c] - protos[i] for i, c in enumerate(classes)],
        dim=0
    )

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)

    overall_results.append((test_subj, fewshot_acc))


print("\nFINAL SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.7539 | best 0.7539
Epoch 02 | zero-shot acc 0.7098 | best 0.7539
Epoch 03 | zero-shot acc 0.7129 | best 0.7539
Epoch 04 | zero-shot acc 0.7256 | best 0.7539
Epoch 05 | zero-shot acc 0.7224 | best 0.7539
Epoch 06 | zero-shot acc 0.7666 | best 0.7666
Epoch 07 | zero-shot acc 0.7529 | best 0.7666
Epoch 08 | zero-shot acc 0.7192 | best 0.7666
Epoch 09 | zero-shot acc 0.6940 | best 0.7666
Epoch 10 | zero-shot acc 0.7308 | best 0.7666
AdaBN + Mahalanobis few-shot acc: 0.839506208896637

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.5250 | best 0.5250
Epoch 02 | zero-shot acc 0.5261 | best 0.5261
Epoch 03 | zero-shot acc 0.5495 | best 0.5495
Epoch 04 | zero-shot acc 0.5346 | best 0.5495
Epoch 05 | zero-shot acc 0.5272 | best 0.5495
Epoch 06 | zero-shot acc 0.5495 | best 0.5495
Epoch 07 | zero-shot acc 0.5122 | best 0.5495
Epoch 08 | zero-shot acc 0.5240 | best 0.5495
Epoch 09 | zero-shot acc 0.5357 | best 0.5

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 5            # 🔥 shorter finetune
BATCH  = 396
LR     = 3e-3         # 🔥 higher LR
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

NBACK_ID = 1

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train,
        y_train, subj_train,
        dataset_id=NBACK_ID
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test, X_plv_test, X_roi_test,
        y_test, subj_test,
        dataset_id=NBACK_ID
    )

    sampler = SubjectBalancedBatchSampler(
        train_ds,
        batch_size=BATCH,
        subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    # 🔥 LOAD PRETRAINED
    model.load_state_dict(torch.load(f"pretrained_fold_{fold_i}.pt"))

    # 🔥 FREEZE EVERYTHING EXCEPT PROJECTOR
    for name, p in model.named_parameters():
        if "projector" not in name:
            p.requires_grad = False

    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=TEMP)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)
    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat(
        [e_support[y_support == c] - protos[i] for i, c in enumerate(classes)],
        dim=0
    )

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)

    overall_results.append((test_subj, fewshot_acc))


print("\nFINAL SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01


ValueError: optimizer got an empty parameter list

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np

EPOCHS = 5
BATCH  = 396
LR     = 5e-4
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

NBACK_ID = 1

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURES
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train,
        y_train, subj_train, dataset_id=NBACK_ID
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test, X_plv_test, X_roi_test,
        y_test, subj_test, dataset_id=NBACK_ID
    )

    sampler = SubjectBalancedBatchSampler(
        train_ds,
        batch_size=BATCH,
        subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    # load pretrained weights (from Phase 1)
    model.load_state_dict(torch.load(f"pretrained_fold_{fold_i}.pt"))

    # =========================
    # FREEZE / UNFREEZE
    # =========================
    for p in model.parameters():
        p.requires_grad = False

    # 🔥 correct for YOUR model
    for p in model.proj.parameters():
        p.requires_grad = True

    # sanity check
    num_trainable = sum(p.requires_grad for p in model.parameters())
    print("Trainable params:", num_trainable)

    opt = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR
    )

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=TEMP)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred.device
            )

            acc = (pred == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)

    # =========================
    # FEW-SHOT (UNCHANGED)
    # =========================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)
    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(idx))
        support_idx.append(idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    e_support = F.normalize(all_e[mask].to(device), dim=1)
    y_support = all_y[mask].to(device)
    e_query   = F.normalize(all_e[~mask].to(device), dim=1)
    y_query   = all_y[~mask].to(device)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support *= weights
    e_query   *= weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("Few-shot acc:", fewshot_acc)

    overall_results.append((test_subj, fewshot_acc))


print("\nFINAL SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Trainable params: 4
Epoch 01 | zero-shot acc 0.7434 | best 0.7434
Epoch 02 | zero-shot acc 0.7434 | best 0.7434
Epoch 03 | zero-shot acc 0.7413 | best 0.7434
Epoch 04 | zero-shot acc 0.7445 | best 0.7445
Epoch 05 | zero-shot acc 0.7445 | best 0.7445
Few-shot acc: 0.7744107842445374

FOLD 2/13 | TEST SUBJECT = subject_02
Trainable params: 4
Epoch 01 | zero-shot acc 0.5165 | best 0.5165
Epoch 02 | zero-shot acc 0.4963 | best 0.5165
Epoch 03 | zero-shot acc 0.4920 | best 0.5165
Epoch 04 | zero-shot acc 0.4984 | best 0.5165
Epoch 05 | zero-shot acc 0.4952 | best 0.5165
Few-shot acc: 0.6769055724143982

FOLD 3/13 | TEST SUBJECT = subject_03
Trainable params: 4
Epoch 01 | zero-shot acc 0.5860 | best 0.5860
Epoch 02 | zero-shot acc 0.5902 | best 0.5902
Epoch 03 | zero-shot acc 0.5892 | best 0.5902
Epoch 04 | zero-shot acc 0.5945 | best 0.5945
Epoch 05 | zero-shot acc 0.5892 | best 0.5945
Few-shot acc: 0.7539682388305664

FOLD 4/13 | TEST SUBJECT = subjec

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

NBACK_ID = 1

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0, 2), keepdim=True)
    time_sd = X_time_train.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train,
        y_train, subj_train, dataset_id=NBACK_ID
    )
    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test, X_plv_test, X_roi_test,
        y_test, subj_test, dataset_id=NBACK_ID
    )

    sampler = SubjectBalancedBatchSampler(
        train_ds,
        batch_size=BATCH,
        subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    # load fold-matched joint pretrain weights
    model.load_state_dict(torch.load(f"pretrained_fold_{fold_i}.pt"))

    # IMPORTANT: do NOT freeze anything
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=TEMP)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0) for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: GATED AdaBN + Mahalanobis
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    # -------- save clean state before BN adaptation --------
    clean_state = copy.deepcopy(model.state_dict())

    # freeze gradients (same as your old setup)
    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    # -------- gated BN decision --------
    # compare support embedding shift before/after BN adaptation
    e_support_pre = all_e_pre[mask].to(device)
    support_shift = e_support_pre.std(dim=0).mean().item()

    # heuristic gate; keep identical style to your gated setup
    # you can tune this threshold later if needed
    BN_SHIFT_THRESHOLD = 0.14
    use_bn = support_shift > BN_SHIFT_THRESHOLD

    if use_bn:
        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)
    else:
        model.load_state_dict(clean_state)

    print("N-back BN USED:", use_bn)

    model.eval()

    # -------- extract embeddings after optional BN adaptation --------
    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # =====================================
    # FEATURE RELIABILITY WEIGHTING
    # =====================================
    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat(
        [e_support[y_support == c] - protos[i] for i, c in enumerate(classes)],
        dim=0
    )

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("N-back gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nN-BACK FINETUNE FROM JOINT PRETRAIN SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.7308 | best 0.7308
Epoch 02 | zero-shot acc 0.7350 | best 0.7350
Epoch 03 | zero-shot acc 0.7434 | best 0.7434
Epoch 04 | zero-shot acc 0.6993 | best 0.7434
Epoch 05 | zero-shot acc 0.7413 | best 0.7434
Epoch 06 | zero-shot acc 0.7150 | best 0.7434
Epoch 07 | zero-shot acc 0.7256 | best 0.7434
Epoch 08 | zero-shot acc 0.7077 | best 0.7434
Epoch 09 | zero-shot acc 0.7024 | best 0.7434
Epoch 10 | zero-shot acc 0.7308 | best 0.7434
N-back BN USED: False
N-back gated AdaBN + Mahalanobis few-shot acc: 0.8114478588104248

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4633 | best 0.4633
Epoch 02 | zero-shot acc 0.4920 | best 0.4920
Epoch 03 | zero-shot acc 0.4601 | best 0.4920
Epoch 04 | zero-shot acc 0.5101 | best 0.5101
Epoch 05 | zero-shot acc 0.4750 | best 0.5101
Epoch 06 | zero-shot acc 0.5548 | best 0.5548
Epoch 07 | zero-shot acc 0.4867 | best 0.5548
Epoch 08 | zero-shot acc 0.4665 | best 0.5548
Epoch 